# Phase 7 — Block design (steps 72–78)

Phases 1–6 explained *why* each block exists. Phase 7 designs them — and the first design step is to decide how much each block is allowed to cost: jitter, power and area (step 72); the second is the loop itself (step 73). This notebook splits the proposal's targets (1.4 ps typ / 3.2 ps max with FFNC, −110/−120 dBc/Hz at 300 kHz, 9 mW, 0.08 mm²) into one allocation per block, shows which allocations are critical, and lets you move them with sliders. The same numbers are in `docs/design/step72_budget.md`; the model is `design/budget/step72_budget.py`. **Device-rule migration (18 Nov 2026):** all analog blocks now use 5 V devices and the output is 50 MHz × 16 = 800 MHz; step 72 has been redone for it (section *Step 72 redo*, `docs/design/step72_hv_budget.md`). Steps 73–78 below are still the 1.6 GHz designs and are being redone in 5 V devices.

**How to use this notebook.** Every step follows the same loop: **What happens** (the observation) →
**How it happens** (the mechanism and the equations, in Razavi's notation) → **Why it happens** (the physical
reason) → an interactive figure → **Try it** (what to change and what to look for) → **Interview questions**
(click a question to open the answer; try to answer it aloud first).

Notation (Razavi, *RF Microelectronics* ch. 9 and *Design of CMOS PLLs*): $\phi_{in}$, $\phi_{out}$ input and output
phase; $\omega_{out} = \omega_0 + K_{VCO} V_{cont}$; $K_{PD}$ phase-detector gain; $I_p$ charge-pump current;
$R_1, C_1, C_2$ loop filter; $N$ divide ratio; $\omega_n$, $\zeta$ natural frequency and damping;
$H(s) = \Phi_{out}/\Phi_{in}$. Frequencies $f$ in Hz, angular $\omega = 2\pi f$ in rad/s.
Design numbers are the submitted proposal's: $f_{REF} = 50$ MHz, $N = 32$, $f_{out} = 1.6$ GHz, $f_c \approx 2.5$ MHz.

## Contents

- [Step 72 — Phase-noise and power budget per block, and the jitter target](#step-72)
- [Step 73 — Loop parameters (f_c, PM, β, R1, C1, C2) from the out2 jitter model](#step-73)
- [Step 74 — VCO redesign: stages, LVT, sizing for frequency, K_VCO and phase noise](#step-74)
- [Step 75 — Current-fed VCO supply (CCO) with PDK-legal resistors](#step-75)
- [Step 76 — REF buffer (REF_IN receiver): low-noise edge](#step-76)
- [Step 77 — Clock generator: sampling clocks, hold pulse, Gm pulse](#step-77)
- [Step 78 — SSPD with hold: sampler, dummy sampler, isolation buffers](#step-78)

*7 steps, 243 interview questions.*

In [ ]:
# setup - run this cell first (Shift+Enter). Figures are interactive: drag the slider under each plot,
# hover for values, drag to zoom, double-click to reset.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sspll_learn as sl
from sspll_learn import P, show, sweep, bode, kT
np.set_printoptions(precision=4, suppress=True)
print("ready - proposal design point:", f"{P['fref']/1e6:.0f} MHz x {P['N']} = {P['f0']/1e9:.2f} GHz")

<a id='step-72'></a>
## Step 72 — Phase-noise and power budget per block, and the jitter target

*Roadmap step 72: budget table*

### What happens
Every block — crystal, reference receiver, SSPD, charge pump, loop-filter resistor, ring, current source, NCB, LDO — adds its
own small random timing error to the output edges. These errors are independent, so the output jitter is their
**root-sum-square**, each shaped by the path from the block to the output. A budget fixes, *before* any transistor is sized,
how much each block may contribute so the sum meets the specification, and gives each block a pass/fail number in its own
units.

### How it happens
Each source enters at a physical place in the loop; its transfer to the output follows from Phases 3–6:

$$S_{out}(f) = \big[N^2 S_{REF} + S_{PD}\big]\big|H + G_c(1-H)\big|^2 + S_{VCO}\big|(1-G_c)(1-H)\big|^2
 + S_{NCB} + (\omega_0 K_s)^2 S_{v,LDO} + |G_c|^2 S_{alias}$$

$$\sigma_t = \frac{1}{2\pi f_0}\sqrt{\int_{10\,\mathrm{kHz}}^{40\,\mathrm{MHz}} S_{out}(f)\,df},\qquad
\sigma_t^2 = \sum_i \sigma_{t,i}^2 .$$

Each block's physics gives its source in its own units, then converted to dBc/Hz:

| Block | Source | Conversion |
|---|---|---|
| REF (XO + receiver) | floor $L_{REF}$ | $\times N^2$ (+30.1 dB); per-edge jitter $\sigma = \sqrt{L f_{REF}}/(2\pi f_{REF})$ |
| SSPD | $kT/C$ | $L = kT/(2 C_s A^2 f_{REF})$ |
| Pulsed-Gm CP | $4kT\gamma g_m$ during duty $d$ | $L = kT\gamma/(2A^2 g_m d)$, with $\beta = 2A g_m d$ |
| Loop-filter $R_1$ | $4kTR_1$ | $\times |C_1/(C_1+C_2+sR_1C_1C_2)|^2\,(K_{VCO}/\omega)^2$ |
| CCO | $i_n$ (A/√Hz) | $S_\varphi = (K_I i_n)^2/f^2$ |
| NCB / LDO | floor / $v_n$ | 1:1 / $(\omega_0 K_s v_n)^2$ |

### Why it happens this way
The reference is a *time* error, so it is multiplied by $N^2$; the SSPD and CP see the VCO directly, so they are not. The
oscillator's noise is high-passed by the loop $(1-H)$ and again by $(1-G_c)$ with FFNC. The NCB and LDO come *after*
everything, so nothing filters them. Aliasing is the price of FFNC: the SSPD folds VCO noise at $f \pm kf_{REF}$ into the band
and the cancellation path adds it to the output. The relative size of the contributions below follows from these facts.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

rows = {}
for c in ("typ", "pess"):
    for on in (False, True):
        rows[(c, on)] = total(c, on)[0]
print(f"{'block':40s} {'typ off':>8s} {'typ on':>8s} {'pess off':>9s} {'pess on':>8s}   (ps rms, 10 kHz-40 MHz)")
for k in rows[("typ", True)]:
    v = [rows[(c, o)].get(k, 0)*1e12 for c in ("typ", "pess") for o in (False, True)]
    print(f"{k:40s} {v[0]:8.3f} {v[1]:8.3f} {v[2]:9.3f} {v[3]:8.3f}")
print(f"{'spec':40s} {'2.3 typ':>8s} {'1.4 typ':>8s} {'4.7 max':>9s} {'3.2 max':>8s}")

**Figure 72.1 — where the jitter comes from** (typ, FFNC off vs on). Hover for values.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

j0, _, _ = total("typ", False); j1, _, _ = total("typ", True)
names = sorted([k for k in j1 if k != "TOTAL"], key=lambda k: j1[k])
fig = go.Figure()
fig.add_trace(go.Bar(y=names, x=[j0.get(k, 0)*1e12 for k in names], orientation="h", name=f"FFNC off ({j0['TOTAL']*1e12:.2f} ps)", marker_color="#2a78d6"))
fig.add_trace(go.Bar(y=names, x=[j1[k]*1e12 for k in names], orientation="h", name=f"FFNC on ({j1['TOTAL']*1e12:.2f} ps)", marker_color="#eb6834"))
fig.update_layout(barmode="group", height=480, xaxis_title="rms jitter contribution (ps)", title="Jitter budget per block, typ")
show(fig)

**Try it.** Square the bars: the ring's $1.87^2 = 3.5$ ps² is 78% of the FFNC-off variance; the SSPD's $0.014^2$ is 0.004%.
A block at one third of the largest adds only 5% to the total — that is why you do not spend effort on small bars.

**Figure 72.2 — the ring 1/f³ corner and the 300 kHz spec.** The proposal specifies −110 dBc/Hz typ at 300 kHz with FFNC off.
Move the ring's 1/f³ corner: at the simulated 1.1 MHz the spec is missed by ~1.5 dB; the budget allocates ≤ 0.6 MHz.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

def frame(fc3):
    tr = []; note = []
    for on, col in ((False, "#2a78d6"), (True, "#eb6834")):
        j, L, _ = total("typ", on, over={"ring VCO core": (-98.35, fc3)})
        tr.append(dict(x=f, y=L, name=f"FFNC {'on' if on else 'off'}", line=dict(color=col)))
        note.append(f"{'on' if on else 'off'}: {j['TOTAL']*1e12:.2f} ps, L(300k) {L[i300]:.1f}")
    tr.append(dict(x=[3e5, 3e5], y=[-110, -120], name="spec typ (off, on)", mode="markers",
                   marker=dict(size=9, color="#59636e", symbol="circle")))
    return tr, f"ring f_c3 = {fc3/1e6:.2f} MHz — " + " | ".join(note)
def lay(fig):
    fig.update_xaxes(type="log", title="offset (Hz)"); fig.update_yaxes(title="L(f) at 1.6 GHz (dBc/Hz)", range=[-140, -95])
show(sweep([0.2e6, 0.3e6, 0.4e6, 0.5e6, 0.6e6, 0.8e6, 1.1e6, 1.5e6, 2.2e6], frame, "ring 1/f³ corner",
           fmt="{:.2g} Hz", active=4, layout=lay))

**Figure 72.3 — loop bandwidth as the fallback knob.** If step 74 cannot lower the corner, `icp_trim` raises $f_c$ (it scales
$\beta$). Watch FFNC-off improve while FFNC-on barely moves (the cancellation already removes the VCO noise near $f_c$), and
remember the price: the sample-and-hold delay costs $\approx 180°\,f_c/f_{REF}$ of phase margin.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

def frame(fc):
    tr = []; note = []
    for on, col in ((False, "#2a78d6"), (True, "#eb6834")):
        j, L, _ = total("typ", on, fc=fc)
        tr.append(dict(x=f, y=L, name=f"FFNC {'on' if on else 'off'}", line=dict(color=col)))
        note.append(f"{'on' if on else 'off'} {j['TOTAL']*1e12:.2f} ps, L300k {L[i300]:.1f}")
    return tr, f"f_c = {fc/1e6:.1f} MHz (hold-delay PM loss {180*fc/50e6:.0f}°) — " + " | ".join(note)
def lay(fig):
    fig.update_xaxes(type="log", title="offset (Hz)"); fig.update_yaxes(title="L(f) (dBc/Hz)", range=[-140, -95])
show(sweep([1.5e6, 2.0e6, 2.5e6, 2.8e6, 3.0e6, 3.5e6, 4.0e6], frame, "loop bandwidth", fmt="{:.2g} Hz", active=2, layout=lay))

**Figure 72.4 — the CCO current noise.** The CCO's thermal floor is fixed by physics: a source of current $I$ with headroom
$V_h$ has $i_n^2 \approx 4kT\cdot 2I/V_h$ → about 7 pA/√Hz at 1.31 mA and 1.2 V. Only its flicker corner is a design choice
(large-area devices, RC-filtered mirror gate). Slide the corner.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

KI = 1.22e12
def frame(fcf):
    tr = []; note = []
    for on, col in ((False, "#2a78d6"), (True, "#eb6834")):
        j, L, S = total("typ", on, over={"CCO current DAC + mirror": (-104.2, fcf)})
        tr.append(dict(x=f, y=10*np.log10(S["CCO current DAC + mirror"]/2), name=f"CCO part, FFNC {'on' if on else 'off'}", line=dict(color=col)))
        note.append(f"{'on' if on else 'off'} total {j['TOTAL']*1e12:.2f} ps (CCO {j['CCO current DAC + mirror']*1e12:.2f})")
    return tr, f"CCO 1/f corner {fcf/1e6:.2f} MHz, i_n = 7.2 pA/√Hz at 1 MHz — " + " | ".join(note)
def lay(fig):
    fig.update_xaxes(type="log", title="offset (Hz)"); fig.update_yaxes(title="CCO contribution at the output (dBc/Hz)", range=[-160, -100])
show(sweep([0.05e6, 0.1e6, 0.2e6, 0.3e6, 0.5e6, 1.1e6, 2.2e6], frame, "CCO flicker corner", fmt="{:.2g} Hz", active=3, layout=lay))

**Figure 72.5 — sensitivity.** Total jitter when one block alone is 3 dB worse than its allocation. Long bars = critical allocations; short bars = slack you can trade for power, area or robustness.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

sens = bud.sensitivity()
base = total("typ", True)[0]["TOTAL"]*1e12
names = sorted(sens, key=lambda k: sens[k][1])
fig = go.Figure(go.Bar(y=names, x=[sens[k][1] - base for k in names], orientation="h", marker_color="#eb6834",
                       name="FFNC on", hovertemplate="%{y}: +%{x:.3f} ps<extra></extra>"))
fig.update_layout(height=430, xaxis_title=f"increase of total typ jitter (ps) over {base:.2f} ps", title="Block +3 dB → total jitter (FFNC on)")
show(fig)

### Power and area

Blocks on the 1.8 V LDO rail draw their current from VDDA = 3.3 V, so every mA there costs **3.3 mW**; the ring is fed from
VDDA through the CCO (4.86 mW). Logic that can run from VDDD costs 1.8 mW/mA — that is why dividers and FLL logic live there.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

tot = {"on": 0, "off": 0, "on_w": 0, "off_w": 0}
print(f"{'block':46s} {'supply':6s} {'I (mA)':>7s} {'P typ':>7s} {'P worst':>8s}")
for name, sup, i, p, pw, ncb in bud.power_table():
    print(f"{name:46s} {sup:6s} {i:7.3f} {p:7.2f} {pw:8.2f}")
    tot["on"] += p; tot["on_w"] += pw
    if not ncb: tot["off"] += p; tot["off_w"] += pw
print(f"TOTAL FFNC on {tot['on']:.2f} mW (worst {tot['on_w']:.2f}; spec 9.0 typ / 15 max)")
print(f"TOTAL FFNC off {tot['off']:.2f} mW (worst {tot['off_w']:.2f}; spec 7.5 typ / 13 max)")
j = total("typ", True)[0]["TOTAL"]
print(f"FoM = {10*np.log10(j**2*tot['on']):.1f} dB (proposal -227.5)")
print("\narea:", ", ".join(f"{n} {a}" for n, a in bud.AREA), f"\n-> {sum(a for _, a in bud.AREA)} um^2 (spec 80,000 typ)")

### Layout convenience, decided now

The budget tells you which nodes are noise-critical, so the schematic can already be drawn the way it will be laid out:

- **Ring (dominant):** 5 identical cells, U-shaped placement so the wrap-around wire matches the others, dummy loads on unused
  taps, integer fingers per device so each cell is one abutting row.
- **CCO (second):** 1:8 mirror as 9 unit devices in a common-centroid row with end dummies; binary DAC from unit cells;
  RC-filtered gate (long high-resistance poly + MOS cap under the array); 3.3 V devices in their own guard ring.
- **Reference receiver (×N²):** short, shielded route from the pad; own VDDA branch; far from the ring.
- **SSPD, CP:** fully differential, mirror-symmetric, unit MIM tiles with a dummy ring, ABBA input pair, non-minimum L.
- **Loop filter:** R1 as equal legs of wide poly; C1 MOS cap with MIM stacked above (~9.5 fF/µm², V_cont > 0.6 V); C2 linear MIM; tiles ≤ 30×30 µm.
- **NCB / LDO:** NCB at the output buffer with its own LDO decap; LDO pass device as unit fingers along the supply edge.
- **Everywhere:** few distinct unit devices; supply domains as bands (VDDA / LDO 1.8 V / VDDD); level shifters only at band edges.

### Interview questions — step 72 (42)

<details><summary><b>Q1.</b> What is a noise budget, and why do it before designing any transistor?</summary>

A table that splits the system target (here 1.4 ps typ / 3.2 ps max rms jitter, −110/−120 dBc/Hz at 300 kHz) into one allocation per block, each in that block's own units (pA/√Hz, nV/√Hz, fs per edge). It turns one system number into pass/fail tests a single block can be designed and simulated against, and it shows *early* which block dominates — so the design effort (and power and area) goes there and not to blocks that cannot matter.

</details>

<details><summary><b>Q2.</b> Why do jitter contributions add in power (rms²) and not linearly?</summary>

The sources are physically independent (different devices), so they are uncorrelated and their variances add: σ² = Σσᵢ². Consequence: a block at half the jitter of the largest one adds only 12% to the total; a block at 1/3 adds 5%. Fixing small contributors is wasted effort.

</details>

<details><summary><b>Q3.</b> Write the output phase-noise PSD of the SSPLL with FFNC in terms of the block sources.</summary>

S_out(f) = [S_REF·N² + S_PD,eq]·|H + G_c(1−H)|² + S_VCO,eq·|(1−G_c)(1−H)|² + S_NCB + S_alias·|G_c|², where H is the closed-loop transfer, G_c the cancellation path (SSPD gain × NCB gain, with sinc hold, half-period delay and pole) and the VCO-like sources (ring, CCO, loop-filter R) all enter at the VCO. Jitter = √(∫S_out df)/(2πf_out), here 10 kHz–40 MHz.

</details>

<details><summary><b>Q4.</b> Why is the reference noise multiplied by N² but the SSPD noise is not?</summary>

The reference sets the *time* of each edge; a time error Δt at f_REF becomes a phase error 2πf_out·Δt at the output, i.e. N times the reference phase error → N² in power. The SSPD compares the VCO directly (no divider), so its voltage noise divided by its gain is already a VCO-phase error: no N. That is the whole point of sub-sampling (−130 vs −110.9 dBc/Hz detector floor at the output, step 8).

</details>

<details><summary><b>Q5.</b> Why does FFNC make the reference contribution *worse* (0.20 → 0.31 ps)?</summary>

With FFNC on, the in-band transfer becomes |H + G_c(1−H)|²: above the loop bandwidth the cancellation path also passes the SSPD's view of the reference, so reference (and SSPD/CP) noise reaches the output up to ≈f_REF/8 instead of f_c. FFNC trades VCO noise for reference noise; it wins because the VCO term is 10× larger.

</details>

<details><summary><b>Q6.</b> Which block dominates the budget, and what does that tell you?</summary>

The ring VCO core (1.87 ps off / 1.13 ps on typ), then the CCO (0.90 / 0.56). Everything else together is < 0.5 ps. So power, area and design time go to the oscillator and its current source; the SSPD, CP and NCB can be designed for linearity, spurs and robustness rather than noise.

</details>

<details><summary><b>Q7.</b> The SSPD kT/C noise contributes only 22 fs. Why not shrink the sampling capacitor to save power?</summary>

Because the cap also sets the charge-sharing/kick-back spur and the hold droop; and the SSPD noise is in-band where FFNC passes it. 100 fF keeps kT/C at −147.8 dBc/Hz (17 dB below the CP allocation), so it is not the limit; going to 25 fF costs +6 dB noise but buys little power (the ring buffers that drive it set the power). The budget says: free to choose the cap for spur and matching, as long as Cs ≥ ~50 fF per side.

</details>

<details><summary><b>Q8.</b> Derive the SSPD kT/C phase-noise floor.</summary>

Each side samples kT/C; the differential sample has v² = 2kT/C. The differential slope is 2A per radian of VCO phase, so one sample carries φ² = (2kT/C)/(2A)² = kT/(2CA²). Samples are independent, so this variance is white over 0…f_REF/2: S_φ = φ²/(f_REF/2) = kT/(C·A²·f_REF), and L = S_φ/2 = kT/(2·C·A²·f_REF) = −147.8 dBc/Hz for C = 100 fF, A = 0.5 V, f_REF = 50 MHz.

</details>

<details><summary><b>Q9.</b> How is the pulsed-Gm charge-pump noise budgeted, and why does the duty cycle matter?</summary>

The transconductor's output noise current 4kTγg_m only reaches the loop filter during the pulse (duty d ≈ 1%, 200 ps of 20 ns). The useful signal current is β = 2A·g_m·d (A per rad). Referred to VCO phase: S_φ = 4kTγg_m·d/β² → L ≈ kTγ/(2A²g_m d) ≈ −140.8 dBc/Hz white, plus a 1/f corner allocated ≤ 300 kHz. Shorter pulses lower the signal more than the noise at equal g_m, so d is set by β and the spur, not by noise.

</details>

<details><summary><b>Q10.</b> Why is the loop-filter resistor a visible contributor (0.28 ps) and what are the knobs?</summary>

Its 4kTR voltage noise (12.6 nV/√Hz for 9.61 kΩ) is filtered by C1/(C1+C2+sRC1C2) and integrated by K_VCO/s, so it behaves like VCO noise (high-passed by 1−H, cancelled by FFNC). Knobs: larger β (more CP current → smaller R, larger C, more area), smaller K_VCO (needs a wider coarse range from the CCO DAC), or a dual-path filter. We accept it: 0.28 ps adds only ~1% to a 2.1 ps total.

</details>

<details><summary><b>Q11.</b> Why must the ring 1/f³ corner be halved (1.1 → 0.6 MHz) in this budget?</summary>

With the simulated 1.1 MHz corner the proposal's own model gives L(300 kHz) = −108.1 dBc/Hz FFNC-off typ, 1.9 dB short of the −110 typ in the specification. At 300 kHz the ring is in its 1/f³ region, so L ∝ (1 + f_c3/f); halving f_c3 plus a lower-flicker CCO brings it to −110.2. Halving needs ~2× gate area in the ring transistors that upconvert flicker, or better rise/fall symmetry (smaller DC term of the ISF, c₀).

</details>

<details><summary><b>Q12.</b> What determines the 1/f³ corner of a ring oscillator?</summary>

Hajimiri–Lee: f_1/f³ = f_1/f(device) · (c₀/c_rms)². c₀ is the DC value of the ISF (zero for a perfectly symmetric waveform), c_rms its rms. So the corner falls with device area (lower device flicker corner) and with symmetry of rise and fall edges (pMOS/nMOS strength balance, equal loading).

</details>

<details><summary><b>Q13.</b> Why is the CCO noise allocated in pA/√Hz, and how do you convert it?</summary>

The CCO is a current source: what it delivers to the ring is a noisy current. Frequency noise = K_I·i_n (K_I = ∂f/∂I = 1.22×10¹² Hz/A), phase noise S_φ = (K_I i_n)²/f², L = S_φ/2. L(1 MHz) = −104.2 dBc/Hz ↔ i_n = 7.2 pA/√Hz.

</details>

<details><summary><b>Q14.</b> Why can't the CCO noise be made arbitrarily small?</summary>

A MOS current source of current I with overdrive (headroom) V_ov has i_n² ≈ 4kTγ·2I/V_ov. At 1.31 mA and ~1.2 V of headroom left by VDDA_min 2.97 V minus the ~1.6–1.8 V ring node, the thermal floor is ≈6–7 pA/√Hz. Degeneration gives the same limit (4kT/R_s with R_s = V_h/I). Only more headroom or less current helps — the budget therefore allocates the thermal floor and attacks only the flicker part.

</details>

<details><summary><b>Q15.</b> Why does a filtered mirror gate help the CCO noise?</summary>

In a 1:8 mirror the reference-side noise (DAC unit cells, IBIAS mirror, diode device) is multiplied by 8 at the output. An RC low-pass on the shared gate (corner ≪ 10 kHz, e.g. a long poly resistor and a MOS cap) removes it inside the integration band, so only the output devices' own noise remains. It costs area, not power.

</details>

<details><summary><b>Q16.</b> Why is the NCB intrinsic noise budget so small (31 fs) yet still important?</summary>

It is added *after* the loop and the cancellation: nothing filters it, so it appears 1:1 at the output with FFNC on, all the way to 40 MHz. −150 dBc/Hz white with a 1 MHz flicker corner gives 31 fs. If the delay line were a long chain of starved inverters, its noise would grow with its delay and could reach hundreds of fs — so the NCB is kept short (2 cells, small delay range).

</details>

<details><summary><b>Q17.</b> How does LDO noise get into the output, and how is that budgeted?</summary>

The NCB delay depends on its supply (≈104 ps/V for 4 inverter edges at ~26 ps/V each): a supply noise density v_n becomes a timing noise K_s·v_n, i.e. phase 2πf_out·K_s·v_n. For ≤ 26 fs contribution the LDO output noise must be ≤ 40 nV/√Hz (10 kHz flicker corner). The ring itself is fed by the high-impedance CCO, not the LDO, so LDO noise does not modulate the oscillator directly.

</details>

<details><summary><b>Q18.</b> What is the aliasing term in the FFNC budget?</summary>

The SSPD samples at f_REF, so VCO noise at f ± k·f_REF folds into the 0…f_REF/2 band (Thijssen: S_s(f) = f_REF Σ S_φ(f − k f_REF)). The cancellation path then *adds* this folded noise to the output (it is not correlated with the output's own noise at f). It costs 0.19 ps typ — the main reason FFNC cannot cancel everything.

</details>

<details><summary><b>Q19.</b> Why is the jitter integrated from 10 kHz to 40 MHz?</summary>

The proposal's specification band. The lower edge excludes slow wander a downstream system tracks; the upper edge is the Nyquist range of a 100 MHz-class sampling clock divided output and where the ring's −20 dB/dec skirt has fallen well below the in-band level. A budget must use the same band as the spec.

</details>

<details><summary><b>Q20.</b> How do you check a budget against the reference range 45–55 MHz?</summary>

Re-run with f_REF = 45 and 55 MHz: f_out scales, the ring's L at 1 MHz scales ~ f_out (same power per edge), the aliasing and the cancellation bandwidth move with f_REF. Typ jitter goes 2.23/1.51 ps at 45 MHz to 2.01/1.23 ps at 55 MHz (off/on) — all inside the maxima.

</details>

<details><summary><b>Q21.</b> What does the 'pessimistic' column represent?</summary>

A coherent worst plausible set for the noise numbers: ring +1 dB and 2× 1/f³ corner, CCO +3 dB and 2× corner, reference receiver +4 dB with a 100 kHz corner, SSPD amplitude 0.35 V, γ = 1.5, NCB −146 dBc/Hz, LDO 80 nV/√Hz and a 30% FFNC gain error. It is not a SPICE corner; it is the budget's stress case that must still meet the *maximum* spec.

</details>

<details><summary><b>Q22.</b> The FFNC-on L(300 kHz) is −119.9 typ against a −120 typ spec. Is that a fail?</summary>

It is a typ number at the limit, not a guaranteed limit (the max is −110, with 4.5 dB margin pess). At 300 kHz with FFNC on the floor is set by the reference ×N²: crystal −155 and receiver −155 → −121.2 dBc/Hz together, before any on-chip noise. Only a cleaner receiver (−157 for ~+0.1 mW) or a better board crystal moves it. This is recorded as a finding.

</details>

<details><summary><b>Q23.</b> How is the power budget built and why does the LDO domain cost more?</summary>

Every block gets a current on a named supply; power = I × supply voltage at the slot. Blocks on the 1.8 V LDO rail draw their current from VDDA = 3.3 V, so each mA costs 3.3 mW, not 1.8 mW: the LDO burns the 1.5 V difference. That is why logic that can live on VDDD (1.8 V, 1.8 mW/mA) is put there.

</details>

<details><summary><b>Q24.</b> Why is the ring fed from VDDA through a CCO instead of from the LDO?</summary>

A current-fed ring has low supply sensitivity (the CCO's output impedance isolates it) and the current DAC also gives the coarse tuning. The price: the ring burns its current at 3.3 V (4.86 mW). Running it from the LDO would cost the same power from VDDA and add the LDO noise directly to the oscillator.

</details>

<details><summary><b>Q25.</b> What is the worst-case power and how is it estimated?</summary>

Ring at 1.76 GHz, ss, 110 °C, VDDA 3.63 V: 7.46 mW (from the proposal's simulated basis). Other blocks: typ current × 1.3 at the maximum supply. Total 13.5 mW FFNC on / 11.4 mW off against 15/13 mW maximum.

</details>

<details><summary><b>Q26.</b> What is the jitter figure of merit and what does this budget give?</summary>

FoM = 10·log₁₀[(σ/1 s)²·(P/1 mW)]. σ = 1.36 ps, P = 9.08 mW → −227.8 dB (proposal −227.5). Note the FoM rewards power and jitter equally in dB: halving the jitter (−6 dB) is worth 4× the power.

</details>

<details><summary><b>Q27.</b> Why does the area table now include the LDO?</summary>

Decision D4 (step 8) added an on-chip LDO for the clean 1.8 V rail; the proposal's area basis listed every block except it. ~4000 µm² (pass device for ~1.2 mA, error amp, compensation and output cap partly as MOS cap). Total 57,500 µm² — still below the 0.08 mm² typ.

</details>

<details><summary><b>Q28.</b> How would you decide between spending 1 mW on a cleaner VCO and 1 mW on a better NCB?</summary>

Use the sensitivity of the total to each block: ring +3 dB moves the typ total from 1.36 to 1.77 ps; NCB +3 dB changes nothing visible. The marginal improvement per mW is far larger at the VCO. A budget makes that comparison numeric.

</details>

<details><summary><b>Q29.</b> What is the 'sensitivity' table and how is it used during design?</summary>

For each block, the total jitter when that block alone is 3 dB worse than allocated. It shows which allocations are critical (ring, CCO, reference) and which have slack (SSPD, CP, NCB, LDO). Slack can be traded for power, area or robustness during the block designs.

</details>

<details><summary><b>Q30.</b> How will each allocation be verified in Phase 8?</summary>

Each block's testbench measures its allocation in its own units: SSPD/CP with .noise and PSS-equivalent transient noise (tb_sspd_char, tb_sscp_*), ring free-running phase noise by transient noise and ISF (tb_vco_*), CCO output current noise by .noise (tb_cco_ac), LDO output noise and PSRR (tb_ldo_ac), NCB delay noise and supply sensitivity (tb_ncb_*). The results replace the allocations in step72_budget.py, which recomputes the total.

</details>

<details><summary><b>Q31.</b> Why keep the budget in a script instead of a spreadsheet?</summary>

Because the transfer functions are frequency-dependent and complex (H, 1−H, G_c, aliasing) — a spreadsheet of single numbers cannot integrate them. A script also re-runs automatically when a simulated curve replaces an allocation, and it is version-controlled next to the schematics.

</details>

<details><summary><b>Q32.</b> What layout decisions follow directly from the noise budget?</summary>

(1) The ring and its CCO get the cleanest, symmetric placement (dominant noise); (2) the CCO output mirror is large-area, common-centroid, with a filtered gate; (3) the reference receiver path is short and shielded (it is ×N²); (4) the SSPD caps are placed symmetrically with dummies (spur, not noise); (5) the NCB is next to the output with its own LDO decap; (6) the loop-filter C1 stacks MIM over MOS to fit the area.

</details>

<details><summary><b>Q33.</b> Why is the loop-filter C1 a MOS capacitor with MIM on top, and what is the catch?</summary>

C1 = 18.2 pF would need ~9100 µm² as MIM alone (~2 fF/µm²), more than the CP+LF area allocation. A thin-oxide MOS cap (7.5 fF/µm² at 0.9 V) with a MIM cap stacked above gives ~9.5 fF/µm² → ~1,900 µm² (~11.5 fF/µm² with both MIM layers). Catch: a MOS cap is non-linear and must be kept in strong inversion/accumulation across the tuning range — acceptable for C1 (the large zero cap), not for C2.

</details>

<details><summary><b>Q34.</b> What is the effect of a loop-gain (K_VCO or β) error on the budget?</summary>

f_c moves (f_c ∝ βK_VCO): a lower f_c lets more VCO noise through and raises FFNC-off jitter; a higher f_c helps here (2.8 MHz closes the 300 kHz spec alone) but costs phase margin through the sample-and-hold delay (≈ 180°·f_c/f_REF). icp_trim[1:0] is the knob to recentre f_c after silicon or after the K_VCO of step 74 is known.

</details>

<details><summary><b>Q35.</b> Why 2× margin between pessimistic budget and the max spec?</summary>

The budget's models (ISF-based ring noise, γ, flicker coefficients) carry several-dB uncertainty, and layout parasitics add noise and power. The proposal says maximum values keep about twice the pre-layout margin; here pess 2.71/1.73 ps vs 4.7/3.2 ps max is 1.7–1.8×.

</details>

<details><summary><b>Q36.</b> Explain 'L = S_φ/2' and why the code uses 2·10^(L/10).</summary>

L(f) is defined as single-sideband noise relative to the carrier; for small phase noise L(f) = S_φ(f)/2 where S_φ is the one-sided phase PSD. Integrating jitter needs S_φ = 2·10^(L/10) rad²/Hz, then σ_t = √∫S_φ df / (2πf_0).

</details>

<details><summary><b>Q37.</b> How does a flicker (1/f) noise source in the CP or SSPD show up at the output?</summary>

Below the loop bandwidth H ≈ 1, so a 1/f input source appears as 1/f at the output — it raises L(f) at low offsets (10–100 kHz). That is why the CP/SSPD allocations include a 300 kHz flicker corner and why their devices are not minimum length.

</details>

<details><summary><b>Q38.</b> What would happen to the budget if the reference were 25 MHz with N = 64?</summary>

The reference term grows by 20·log(2) = 6 dB (N²), the SSPD term grows because A²·f_REF halves (+3 dB), the cancellation bandwidth halves (f_REF/8) and aliasing worsens — the 50 MHz reference is a deliberate budget choice.

</details>

<details><summary><b>Q39.</b> Which allocations are measured on silicon, and how?</summary>

Total jitter and L(f) on CLK_OUT (/16 + 24.1 dB), FFNC on vs off on the same chip, supply sensitivity with the VTEST tone, and power per domain from slot currents with blocks enabled in turn. Individual block noise is not directly observable — the budget lets you attribute the measured total.

</details>

<details><summary><b>Q40.</b> A colleague proposes a 3-stage ring to save power. What does the budget say?</summary>

The step-6 study: 3 × 1.0 at 0.61 mA gives −87.4 dBc/Hz vs −94.2 for 5 × 1.5 — 6.8 dB worse at the dominant block; jitter would roughly double. Saving ~2 mW does not pay; the 5-stage verdict stands.

</details>

<details><summary><b>Q41.</b> Why is the FLL charge pump nearly absent from the noise budget?</summary>

It has a dead zone: in lock the phase error sits inside it and the FLL CP is off — no noise current reaches the loop filter. Its leakage and switch charge injection matter for spurs (step 73), not for random jitter.

</details>

<details><summary><b>Q42.</b> What are the three numbers every block designer must take away from step 72?</summary>

Its noise allocation (in its own units), its supply current on a named supply, and its area. Plus its layout rules from the budget: which nodes are noise-critical, which must be symmetric and which supply domain it lives in.

</details>

<a id='step-72-hv'></a>
### Step 72 redo — 800 MHz with 5 V analog devices (18 Nov 2026)

**What happens.** The challenge rules changed: every analog block must use the 5 V devices (`nfet/pfet_g5v0d10v5`) and
only logic may use 1.8 V devices. A full-swing 3-stage ring in 5 V devices tops out at about 1.1 GHz at ss 110 °C, so the
output moved to **50 MHz × 16 = 800 MHz** and the regulated 1.8 V rail (the LDO) was dropped. The budget is redone with the
same model; the surprise is that the jitter does not change.

**How it happens.** A ring's phase noise per milliwatt is described by its figure of merit:

$$\text{FoM}_{ring} = \mathcal{L}(\Delta f) + 10\log\frac{P}{1\,\text{mW}} - 20\log\frac{f_0}{\Delta f}$$

At fixed power and FoM, $\mathcal{L} \propto f_0^2$: 6.02 dB lower at half the carrier. Jitter divides the integrated phase
noise by $(2\pi f_0)^2$, so the carrier cancels:

$$\sigma_{t,ring}^2 = \frac{\int 2\cdot 10^{\mathcal{L}(\Delta f)/10}\, d\Delta f}{(2\pi f_0)^2} \;\propto\; \frac{1}{P}$$

Without the LDO the NCB and the ring's current source see the raw VDDA. Two new terms replace the LDO term — one adds at
the output (the NCB delay moves by $K_s$ per volt), one enters as VCO noise (the ring frequency moves by $K_{vdd}$ per volt
through the CCO's finite output impedance):

$$S_{\varphi,NCB}(f) = (2\pi f_0 K_s)^2\, S_{v,DD}(f), \qquad
S_{\varphi,ring}(f) = \frac{K_{vdd}^2\, S_{v,DD}(f)}{f^2}\,\big|(1-G_c)(1-H)\big|^2$$

**Why it happens this way.** Timing jitter is a time error; frequency only converts it to radians and back. Thick-oxide
devices are slow (f_T 10.9 GHz vs 58 GHz) but *quieter* per unit $g_m$ — their long channels have little excess thermal
noise (γ ≈ 1.1 instead of 1.9) — so a 5 V ring at the step-74 power can reach the step-74 FoM. The budget assumes it does
(typ) and also covers missing it by 3 dB with a doubled 1/f³ corner (pess). Full write-up:
`docs/design/step72_hv_budget.md`; model: `design/budget/step72_hv_budget.py`.

In [ ]:
import sys, json; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_hv_budget as hv
fh = pm.freq_axis(n=700)
def total_hv(corner, on, **kw):
    ff, S = hv.spectra(corner, on, f=fh, **kw)
    j, L = hv.jitter(S, ff)
    return j, L, S
j300 = int(abs(fh - 3e5).argmin())

rows = {(c, o): total_hv(c, o)[0] for c in ("typ", "pess") for o in (False, True)}
print(f"{'block':44s} {'typ off':>8s} {'typ on':>8s} {'pess off':>9s} {'pess on':>8s}   (ps rms, 10 kHz-40 MHz, 800 MHz)")
for k in rows[("typ", True)]:
    v = [rows[(c, o)].get(k, 0)*1e12 for c in ("typ", "pess") for o in (False, True)]
    print(f"{k:44s} {v[0]:8.3f} {v[1]:8.3f} {v[2]:9.3f} {v[3]:8.3f}")
print(f"{'proposal target':44s} {'2.6 typ':>8s} {'1.6 typ':>8s} {'5.0 max':>9s} {'3.2 max':>8s}")

**Figure 72.6 — where the jitter comes from at 800 MHz** (typ). Compare with Figure 72.1: the bars are almost the same length in picoseconds, the LDO bar is gone and two VDDA bars have appeared.

In [ ]:
import sys, json; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_hv_budget as hv
fh = pm.freq_axis(n=700)
def total_hv(corner, on, **kw):
    ff, S = hv.spectra(corner, on, f=fh, **kw)
    j, L = hv.jitter(S, ff)
    return j, L, S
j300 = int(abs(fh - 3e5).argmin())

j0, _, _ = total_hv("typ", False); j1, _, _ = total_hv("typ", True)
names = sorted([k for k in j1 if k != "TOTAL"], key=lambda k: max(j1[k], j0.get(k, 0)))
fig = go.Figure()
fig.add_trace(go.Bar(y=names, x=[j0.get(k, 0)*1e12 for k in names], orientation="h", name=f"FFNC off ({j0['TOTAL']*1e12:.2f} ps)", marker_color="#2a78d6"))
fig.add_trace(go.Bar(y=names, x=[j1[k]*1e12 for k in names], orientation="h", name=f"FFNC on ({j1['TOTAL']*1e12:.2f} ps)", marker_color="#eb6834"))
fig.update_layout(barmode="group", height=500, xaxis_title="rms jitter contribution (ps)", title="Jitter budget per block, typ, 800 MHz")
show(fig)

**Figure 72.7 — what if the thick-oxide ring misses the FoM?** Slide the penalty on the ring's thermal noise (0 dB = the
step-74 FoM at the same power). The proposal's maximum (3.2 ps with FFNC on) still holds at 6 dB (2.40 ps); the typical
1.6 ps holds up to a 2 dB penalty (1.62 ps), and the FFNC-off typical 2.6 ps likewise (2.56 ps).

In [ ]:
import sys, json; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_hv_budget as hv
fh = pm.freq_axis(n=700)
def total_hv(corner, on, **kw):
    ff, S = hv.spectra(corner, on, f=fh, **kw)
    j, L = hv.jitter(S, ff)
    return j, L, S
j300 = int(abs(fh - 3e5).argmin())

RING = "ring VCO core (5 V, 3-stage)"
L0, fc3 = hv.ALLOC[RING]["typ"]
def frame(pen):
    tr = []; note = []
    for on, col in ((False, "#2a78d6"), (True, "#eb6834")):
        j, L, _ = total_hv("typ", on, over={RING: (L0 + pen, fc3)})
        tr.append(dict(x=fh, y=L, name=f"FFNC {'on' if on else 'off'}", line=dict(color=col)))
        note.append(f"{'on' if on else 'off'}: {j['TOTAL']*1e12:.2f} ps, L(300k) {L[j300]:.1f}")
    tr.append(dict(x=[3e5, 3e5], y=[-114, -124], name="spec typ (off, on)", mode="markers",
                   marker=dict(size=9, color="#59636e", symbol="circle")))
    return tr, f"ring FoM penalty {pen:.0f} dB (L(1 MHz) = {L0 + pen:.1f} dBc/Hz) — " + " | ".join(note)
def lay(fig):
    fig.update_xaxes(type="log", title="offset (Hz)"); fig.update_yaxes(title="L(f) at 800 MHz (dBc/Hz)", range=[-145, -95])
show(sweep([0, 1, 2, 3, 4, 5, 6], frame, "ring FoM penalty", fmt="{:.0f} dB", active=0, layout=lay))

**Figure 72.8 — why the CCO must be cascoded.** Without an LDO, VDDA noise reaches the ring through the CCO's output
impedance. Here VDDA carries a poor 300 nV/√Hz; slide the ring's supply pushing $K_{vdd}$. At 5 MHz/V (a cascoded,
long-channel CCO) it adds 0.29 ps with FFNC off; at 20 MHz/V (1.17 ps) it is larger than the CCO's own noise (0.90 ps), and at
40 MHz/V (2.35 ps, a simple mirror) larger than the ring itself.

In [ ]:
import sys, json; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_hv_budget as hv
fh = pm.freq_axis(n=700)
def total_hv(corner, on, **kw):
    ff, S = hv.spectra(corner, on, f=fh, **kw)
    j, L = hv.jitter(S, ff)
    return j, L, S
j300 = int(abs(fh - 3e5).argmin())

SUP = "VDDA noise into the ring (CCO r_out)"
def frame(k):
    tr = []; note = []
    for on, col in ((False, "#2a78d6"), (True, "#eb6834")):
        j, L, S = total_hv("typ", on, over={SUP: (300e-9, k*1e6)})
        tr.append(dict(x=fh, y=10*np.log10(np.maximum(S[SUP], 1e-30)/2), name=f"VDDA part, FFNC {'on' if on else 'off'}", line=dict(color=col)))
        note.append(f"{'on' if on else 'off'} total {j['TOTAL']*1e12:.2f} ps (VDDA {j[SUP]*1e15:.0f} fs)")
    return tr, f"K_vdd = {k:g} MHz/V at 300 nV/√Hz — " + " | ".join(note)
def lay(fig):
    fig.update_xaxes(type="log", title="offset (Hz)"); fig.update_yaxes(title="VDDA contribution (dBc/Hz)", range=[-170, -100])
show(sweep([1, 2, 5, 10, 20, 40], frame, "K_vdd", fmt="{:g} MHz/V", active=2, layout=lay))

**The device data behind the allocations.** `design/feas/hv_noise.py` biases each device at $g_m/I_D = 10$, 100 µA and
simulates its drain-current noise with ngspice `.noise`. Read γ (thermal noise per $g_m$, lower is better) and the 1/f
corner (where flicker equals thermal).

In [ ]:
import json
d = json.load(open("../design/feas/hv_noise.json"))
print(f"{'device':16s} {'L':>5s} {'W':>7s} {'gamma':>6s} {'1/f corner':>11s} {'vn(10k)':>12s}")
for r in d:
    print(f"{r['model']:16s} {r['l']:5.2f} {r['w']:7.1f} {r['gamma']:6.2f} {r['corner_hz']/1e6:8.2f} MHz {r['vn_10k']*1e9:7.1f} nV/√Hz")

**Power at 800 MHz.** Everything analog now runs from VDDA = 3.3 V (3.3 mW per mA); logic from VDDD = 1.8 V. The LDO is gone
(−0.48 mW); level shifters (+0.33 mW) and the output divider (+0.12 mW) arrive. The total stays at about 9 mW.

In [ ]:
import sys, json; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_hv_budget as hv
fh = pm.freq_axis(n=700)
def total_hv(corner, on, **kw):
    ff, S = hv.spectra(corner, on, f=fh, **kw)
    j, L = hv.jitter(S, ff)
    return j, L, S
j300 = int(abs(fh - 3e5).argmin())

tot = {"on": 0, "off": 0, "on_w": 0, "off_w": 0}
for name, sup, i, p, pw, ncb in hv.power_table():
    print(f"{name:74s} {sup:5s} {i:6.3f} mA {p:6.2f} mW (worst {pw:5.2f})")
    tot["on"] += p; tot["on_w"] += pw
    if not ncb: tot["off"] += p; tot["off_w"] += pw
j = total_hv("typ", True)[0]["TOTAL"]
print(f"TOTAL FFNC on {tot['on']:.2f} mW (worst {tot['on_w']:.2f}); off {tot['off']:.2f} mW (worst {tot['off_w']:.2f})")
print(f"FoM = {10*np.log10(j**2*tot['on']):.1f} dB; area {sum(a for _, a in hv.AREA)} um^2")

### Interview questions — step 72 — 800 MHz redo (8)

<details><summary><b>Q1.</b> Why is a ring oscillator's timing jitter independent of its frequency at fixed power and FoM, while its phase noise in dBc/Hz is not?</summary>

In the 1/f² region L(Δf) ≈ (2kT/P)·η·(V_DD/V_char)·(f₀/Δf)², so at fixed P and FoM L ∝ f₀². Jitter is the integral of the phase PSD divided by (2πf₀)², which removes the f₀² again: σ_t² ∝ 1/P. Halving the carrier (1.6 → 0.8 GHz) lowers L(1 MHz) by 6.02 dB and leaves the seconds unchanged — which is why the 800 MHz budget gives 2.12 / 1.37 ps, the same as the 1.6 GHz budget (2.11 / 1.36 ps), at the same ring power.

</details>

<details><summary><b>Q2.</b> At N = 16 the reference noise is multiplied by N² = 256 instead of 1024. Does the reference contribute less jitter?</summary>

No. N² converts a reference *time* error into output *phase*: φ_out = 2πf_out·Δt. Halving f_out halves the phase for the same Δt, so the dBc/Hz drop by 6 dB, but dividing by (2πf_out)² to get seconds undoes it. A timing error at the reference stays the same timing error at the output (0.20 ps off / 0.31 ps on here, as before).

</details>

<details><summary><b>Q3.</b> Thick-oxide (5 V) devices are about 5× slower. How can they give the same or better ring phase noise per milliwatt?</summary>

Phase noise per milliwatt is set by the noise current per unit g_m (γ) and by the swing relative to the noise (V_char), not by f_T. hv_noise.py finds γ = 1.10 for the 5 V nMOS and 0.53 for the 5 V pMOS against 1.87 and 1.22 for the 1.8 V devices: 2.3–3.6 dB less thermal noise per g_m, because the long channel has no short-channel excess noise. Speed only limits how high the ring can go (1.1 GHz at ss 110 °C), which is why the carrier moved to 800 MHz.

</details>

<details><summary><b>Q4.</b> What does γ mean, and why is it larger in short-channel devices?</summary>

γ is the ratio of the drain thermal-noise current to 4kT·g_m: S_id = 4kTγg_m. A long-channel device in saturation has γ ≈ 2/3. In short channels the high lateral field heats the carriers and velocity saturation lowers g_m for the same channel noise, so γ rises to 1.5–2.5. The 0.15 µm 1.8 V nMOS shows 1.87; the 0.5 µm 5 V nMOS, with a thick oxide and a weaker field, shows 1.10.

</details>

<details><summary><b>Q5.</b> Why does VDDA noise through the NCB add directly to the output, while VDDA noise into the ring is cancelled by the FFNC?</summary>

The NCB is the last block: a supply change shifts its delay by K_s·Δv, an output timing error that nothing after it can see — S_φ = (2πf₀K_s)²S_v. A supply change in the ring changes its *frequency* (K_vdd·Δv), so it integrates to phase like any VCO noise; the loop high-passes it and the SSPD measures the resulting phase error, which the FFNC then removes.

</details>

<details><summary><b>Q6.</b> How would you measure K_vdd and K_s in simulation, and why must K_vdd stay small although the loop high-passes it?</summary>

K_vdd: run the free-running ring at two VDDA values (or a small sine on VDDA) and take Δf/ΔV (tb_vco_kvdd). K_s: drive the NCB with a fixed input and measure the output-edge delay at two VDDA values (Δt/ΔV). K_vdd matters because the noise it brings in is VCO-like and grows as 1/f²: at 20 MHz/V and 300 nV/√Hz it adds 1.17 ps with FFNC off — as much as the CCO. A cascoded CCO with L ≥ 1 µm keeps it at 5 MHz/V (0.10 ps).

</details>

<details><summary><b>Q7.</b> Going from L = 0.5 µm to 1 µm cuts the 5 V nMOS 1/f corner by 20×, though the gate area grows only about 6×. Why?</summary>

At constant g_m/I_D and current, the longer device is wider and its gate area W·L grows ~6×, which lowers the flicker PSD ~6×. But the corner frequency is flicker divided by thermal, and thermal per g_m stays about the same, so area alone gives ~6×. The rest comes from the model: SKY130's flicker parameters depend on the length bin (short devices have more edge and halo-implant traps per area). Lesson: for current sources, take the length the model rewards and measure it — hv_noise.py is that measurement.

</details>

<details><summary><b>Q8.</b> Why does the SSPD gain A per VCO radian double at half the carrier, and what does that do to its kT/C noise?</summary>

A = (edge slope)/ω₀: the sampler converts a timing error into a voltage through the slope, and one radian at 800 MHz is twice as long in time as at 1.6 GHz. With the same slope, A doubles. The kT/C term L = kT/(2C_sA²f_REF) falls by 6 dB in rad² — the same timing error in seconds — and stays negligible (0.04 ps).

</details>

<a id='step-73'></a>
## Step 73 — Loop parameters (f_c, PM, β, R1, C1, C2) from the out2 jitter model

*Roadmap step 73: design sheet*

### What happens
The loop filter and the charge-pump gain decide how the work is split: **below $f_c$** the loop makes the VCO follow the
clean reference, **above $f_c$** the VCO runs free and FFNC removes its noise up to about $f_{REF}/8$. Step 73 picks
$f_c$ and PM from the jitter model, computes $R_1, C_1, C_2$ for the *real, sampled* loop, and turns them into PDK devices
in whole layout tiles.

### How it happens
$$L(s) \approx \beta K_{VCO}\frac{Z(s)}{s}e^{-st_p},\quad Z(s) = \frac{1+sR_1C_1}{s(C_1+C_2)(1+sR_1\frac{C_1C_2}{C_1+C_2})},
\quad \beta = 2A\,g_m\frac{\tau}{T}$$

Closed-form start (zero and pole around $\omega_c$, spread $b=\tan(45°+\varphi_{lead}/2)$):
$C_{tot} = b\beta K_{VCO}/\omega_c^2$, $C_2 = C_{tot}/b^2$, $C_1 = C_{tot}-C_2$, $R_1 = b/(\omega_c C_1)$, with
$\varphi_{lead} = PM + 360°f_c t_p$. Then the **exact sampled model** corrects it: the SSPD samples once per period and the
Gm delivers one packet $q[k] = \beta T\theta_e[k]$ a delay $t_p$ later, so
$$x[k+1] = e^{AT}x[k] + e^{A(T-t_p)}B\,q[k],\qquad L(z) = \beta T\,C(zI-e^{AT})^{-1}e^{A(T-t_p)}B .$$

### Why it happens this way
Sampling and the packet delay cost phase that a continuous model does not see (it would give 40.6° instead of 50° at
$t_p$ = 11 ns). The loop sees only the product $\beta K_{VCO}$, so the big passives can be frozen now and the unknown
$K_{VCO}$ absorbed later by the small $g_m$ cells. A MOS capacitor is non-linear, so $C_1$ can be a dense MOS cap only
if $V_{cont}$ stays where it is flat.

**Figure 73.1 — what pins $f_c$.** Slide $f_c$ (PM 50°). Watch three numbers in the title: out2 jitter (barely moves),
out1 $L$(300 kHz) (needs $f_c \ge 2.5$ MHz for −110 dBc/Hz) and the FFNC jitter reduction (needs $f_c \le 2.5$ MHz for
35 %). Only 2.5 MHz meets both typ values.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

def frame(fc):
    tr = []; jj = {}
    for on, col in ((False, "#2a78d6"), (True, "#eb6834")):
        j, L, _ = total("typ", on, fc=fc)
        jj[on] = (j["TOTAL"]*1e12, L[i300])
        tr.append(dict(x=f, y=L, name=f"{'out2 (FFNC on)' if on else 'out1 (FFNC off)'}", line=dict(color=col)))
    red = 100*(1 - jj[True][0]/jj[False][0])
    return tr, (f"f_c {fc/1e6:.1f} MHz: out1 {jj[False][0]:.2f} ps, L300k {jj[False][1]:.1f} | out2 {jj[True][0]:.2f} ps | "
                f"FFNC reduction {red:.0f} %")
def lay(fig):
    fig.update_xaxes(type="log", title="offset (Hz)"); fig.update_yaxes(title="L(f) (dBc/Hz)", range=[-140, -95])
show(sweep([1.5e6, 2.0e6, 2.5e6, 3.0e6, 3.5e6, 4.0e6, 5.0e6], frame, "f_c", fmt="{:.2g} Hz", active=2, layout=lay))

**Figure 73.2 — the packet delay $t_p$ eats phase margin.** The filter is fixed (designed for $t_p$ = 2.5 ns). Slide
$t_p$ and watch the phase at crossover. The dashed line is the continuous model, which cannot see the delay at all.

In [ ]:
import sys, json, csv; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget"); sys.path.insert(0, "../design/loop")
import sampled_loop as sl
D = json.load(open("../design/loop/data/step73_design.json"))
R, C1, C2, KV, beta, tp, T = D["R"], D["C1"], D["C2"], D["KV"], D["beta"], D["tp"], 20e-9

f = np.logspace(4, np.log10(0.49/T), 400)
s = 2j*np.pi*f
Lc = beta*KV*(1 + s*R*C1)/(s*(C1 + C2)*(1 + s*R*C1*C2/(C1 + C2)))/s
def frame(t):
    L = sl.loop_gain(f, R, C1, C2, KV, beta, T, t)
    fc, pm_ = sl.margins(R, C1, C2, KV, beta, T, t)
    ph = np.degrees(np.unwrap(np.angle(L))); ph -= 360*np.round(ph[0]/360)
    return ([dict(x=f, y=ph, name="sampled loop ∠L", line=dict(color="#2a78d6")),
             dict(x=f, y=np.degrees(np.unwrap(np.angle(Lc))), name="continuous ∠L", line=dict(color="#eb6834", dash="dash"))],
            f"t_p = {t*1e9:.1f} ns: f_c = {fc/1e6:.2f} MHz, PM = {pm_:.1f}°  (delay lag at f_c = {360*fc*t:.1f}°)")
def lay(fig):
    fig.update_xaxes(type="log", title="frequency (Hz)"); fig.update_yaxes(title="phase of L (deg)", range=[-260, -100])
    fig.add_vline(x=D["fc"], line_width=1, line_color="#59636e")
show(sweep([0.5e-9, 1e-9, 2.5e-9, 5e-9, 8e-9, 11e-9], frame, "packet delay t_p", fmt="{:.2g} s", active=2, layout=lay))

**Figure 73.3 — loop-gain spread and `icp_trim`.** $m = (A g_m \tau K_{VCO})$/nominal varies over PVT. Slide $m$ and watch
the phase-step response (cycle by cycle, exact): small $m$ → slow and under-damped, large $m$ → fast but ringing. The
`icp_trim` codes multiply $\beta$ by 0.67 / 1 / 1.33 / 1.67 to pull $m$ back towards 1.

In [ ]:
import sys, json, csv; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget"); sys.path.insert(0, "../design/loop")
import sampled_loop as sl
D = json.load(open("../design/loop/data/step73_design.json"))
R, C1, C2, KV, beta, tp, T = D["R"], D["C1"], D["C2"], D["KV"], D["beta"], D["tp"], 20e-9

def frame(m):
    t, e, v = sl.step(R, C1, C2, KV, beta*m, T, tp, ncyc=150, phi_step=1.0)
    fc, pm_ = sl.margins(R, C1, C2, KV, beta*m, T, tp)
    return ([dict(x=t*1e6, y=e/(2*np.pi*1.6e9)*1e12, name="phase error", line=dict(color="#2a78d6", shape="hv")),
             dict(x=[0, 3], y=[10, 10], name="±10 ps", line=dict(color="#59636e", dash="dot")),
             dict(x=[0, 3], y=[-10, -10], name="−10 ps", line=dict(color="#59636e", dash="dot"), showlegend=False)],
            f"m = {m:.2f}: f_c = {fc/1e6:.2f} MHz, PM = {pm_:.1f}°")
def lay(fig):
    fig.update_xaxes(title="time after a 1 rad phase step (µs)", range=[0, 3]); fig.update_yaxes(title="phase error (ps)", range=[-60, 100])
show(sweep([0.35, 0.5, 0.67, 0.8, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0], frame, "loop-gain multiplier m", fmt="{:.2f}", active=4, layout=lay))

**Figure 73.4 — why $V_{cont}$ must stay in 0.6–1.2 V.** The measured C-V of the nfet_01v8 MOS capacitor (ngspice, frozen
PDK) at three temperatures and three corners. Below about 0.6 V the channel disappears and $C_1$ collapses — the zero
would move up and the phase margin would fall.

In [ ]:
import csv
rows = list(csv.DictReader(open("../design/loop/data/moscap_cv.csv")))
fig = go.Figure()
cols = {"tt": "#2a78d6", "ss": "#eb6834", "ff": "#1baf7a"}; dash = {"-40": "dot", "27": "solid", "110": "dash"}
for c in ("tt", "ss", "ff"):
    for t in ("-40", "27", "110"):
        r = [x for x in rows if x["corner"] == c and x["temp"] == t]
        fig.add_trace(go.Scatter(x=[float(x["vg"]) for x in r], y=[float(x["c_mos_fF"])/100 for x in r], name=f"{c}, {t} °C",
                                 line=dict(color=cols[c], dash=dash[t], width=2 if t == "27" else 1.2)))
fig.add_vrect(x0=0.6, x1=1.2, fillcolor="#e6e8eb", opacity=0.5, line_width=0, annotation_text="V_cont window")
fig.update_layout(height=420, title="nfet_01v8 gate capacitance density (W = L = 10 µm)",
                  xaxis_title="V_gate (V)", yaxis_title="C (fF/µm²)")
show(fig)

### The design sheet

| Parameter | Value |
|---|---|
| $f_c$, PM | 2.50 MHz, 50.0° (exact sampled loop, packet delay $t_p$ = 2.5 ns, $\le$ 5 ns required) |
| $\beta$ | 0.848 µA/rad = $2A g_m d$ with $A$ = 0.5 V, $d$ = 1 %, $g_m$ = 84.8 µS = 3 units × 28.3 µS |
| $R_1$ | 11.01 kΩ — res_xhigh_po_2p85, L = 15.5 µm |
| $C_1$ | 17.11 pF — 2 tiles × (9 nfet_01v8 W = L = 10 µm + 1 cap_mim_m3_1 30×30 µm on top) |
| $C_2$ | 2.205 pF — 2 × cap_mim_m3_1 30 × 18.1 µm |
| zero / pole | 0.845 / 7.40 MHz; peaking 2.5 dB; $f_{3dB}$ 4.3 MHz; step settles in 0.34 µs |
| $V_{cont}$ window | 0.6–1.2 V |
| `icp_trim` | 00 → ×1, 01 → ×1.33, 10 → ×0.67, 11 → ×1.67 (Gm unit count 3 / 4 / 2 / 5) |
| residual spread after trim | $m$ within ×0.75–×1.3 → PM 45.5–50° at every R/C corner |

**Layout convenience built into these numbers:** $C_1$ is an integer number of identical tiles (β was rescaled to fit
the tiles, not the other way round); $C_2$ is two equal MIM rectangles; $R_1$ is one body of a legal width; the trim is
done with identical Gm unit cells, not with odd-sized devices; the MIM over the MOS cap shields the filter node from
routing above. Full sheet with every number: `docs/design/step73_loop.md`.

### Interview questions — step 73 (36)

<details><summary><b>Q1.</b> Why is the loop bandwidth chosen from the out2 (FFNC-on) jitter model and not from out1?</summary>

out2 is the output the specification is written for (1.4 ps typ). But out2 is almost flat in f_c (1.34–1.36 ps from 1.5 to 5 MHz) because FFNC cancels the VCO noise the loop would otherwise have to suppress. So out2 says 'anything in 1.5–5 MHz works', and the *other* specs pick the point inside that range.

</details>

<details><summary><b>Q2.</b> Then what actually pins f_c = 2.5 MHz?</summary>

Two specs pull in opposite directions. The FFNC-off phase noise at 300 kHz (−110 typ) needs f_c ≥ 2.5 MHz, because a wider loop suppresses more ring noise. The FFNC jitter reduction (35 % typ, 20 % min) needs f_c ≤ 2.5 MHz, because a wider loop leaves less VCO noise for FFNC to remove. 2.5 MHz is the only point that meets both typ values.

</details>

<details><summary><b>Q3.</b> Why does a wider loop reduce the benefit of FFNC?</summary>

FFNC removes VCO noise between f_c and about f_REF/8. If the loop already suppresses the VCO up to a higher f_c, the band where FFNC helps shrinks: at 4 MHz the reduction is 18 % (below the 20 % minimum), at 5 MHz only 9 %.

</details>

<details><summary><b>Q4.</b> Why 50° of phase margin and not 60°?</summary>

In the jitter model a higher PM lowers peaking and jitter slightly (1.36 → 1.30 ps at 60°), but the zero moves lower (the zero–pole spread b grows), which weakens low-frequency VCO suppression: L(300 kHz) FFNC-off goes from −110.2 to −108. 50° is the highest PM that still meets −110, and it keeps the overshoot (27 %) and peaking (2.5 dB) moderate.

</details>

<details><summary><b>Q5.</b> What is the 'sampled-data' loop and why is it needed here?</summary>

The SSPD looks at the phase only once per reference period and the pulsed Gm delivers one charge packet per period, a delay t_p after the sampling edge. A continuous-time model ignores both. The exact model propagates the filter + VCO state over one period with the matrix exponential, x[k+1] = e^{AT}x[k] + e^{A(T−t_p)}B·q[k], and evaluates L(e^{jωT}). It shows the continuous design would land at 48.7° (t_p = 0.5 ns) or 40.6° (t_p = 11 ns) instead of 50°.

</details>

<details><summary><b>Q6.</b> How large is the phase lag of the packet delay, and how do you compensate it?</summary>

A delay t_p adds 360°·f_c·t_p of lag at crossover: 2.3° at 2.5 ns, 9.9° at 11 ns. It is compensated by designing the filter for a larger lead (52.7° instead of 50° at t_p = 2.5 ns), i.e. a wider zero–pole spread, then iterating on the exact sampled model until f_c and PM land exactly.

</details>

<details><summary><b>Q7.</b> Why does t_p become a requirement on the clock generator?</summary>

With the filter fixed, PM falls from 51.3° (t_p = 0.5 ns) to 48.3° (5 ns) and 43.4° (11 ns). Firing the Gm pulse right after the hold transfer instead of half a period later is free in power and area, so step 77 must deliver t_p ≤ 5 ns (2.5 ns nominal).

</details>

<details><summary><b>Q8.</b> Write β for the pulsed-Gm charge pump and explain each term.</summary>

β = 2A·g_m·d (A per rad of VCO phase). 2A is the differential SSPD slope (V/rad) for a sampled amplitude A, g_m converts the held voltage into current, and d = τ/T is the fraction of the period the Gm conducts. The average current into the filter is β·θ_e. Here A = 0.5 V, d = 0.01 (200 ps / 20 ns), g_m = 84.8 µS → β = 0.848 µA/rad.

</details>

<details><summary><b>Q9.</b> Why can K_VCO still be unknown when the filter is frozen?</summary>

The loop depends only on the product β·K_VCO (C_tot = b·β·K_VCO/ω_c², R = b/(ω_c C1)). If step 74 lands a different K_VCO, g_m is rescaled (g_m ∝ 1/K_VCO) and the filter is unchanged. The large-area parts (C1, C2) are frozen now; the small-area part (Gm unit size) absorbs the uncertainty. That is a layout-driven decision.

</details>

<details><summary><b>Q10.</b> How were C1 and β picked together?</summary>

C1 was snapped to whole layout tiles first (2 tiles of 8.56 pF), then β was scaled by the same factor (×0.848) so β·K_VCO/C stays the same, and R (×1/0.848) and C2 (×0.848) followed. The loop is identical; the layout gets an integer number of identical tiles.

</details>

<details><summary><b>Q11.</b> What is in one C1 tile and why?</summary>

9 nfet_01v8 MOS-capacitor units (W = L = 10 µm, 3×3) with one 30×30 µm cap_mim_m3_1 above them: 6.73 pF + 1.82 pF = 8.56 pF at V_cont = 0.9 V in about 34×34 µm. The MOS cap gives density (7.5 fF/µm²), the MIM on top is free extra capacitance in the same footprint and shields the gate from routing above.

</details>

<details><summary><b>Q12.</b> What is the catch with a MOS capacitor in the loop filter?</summary>

Its capacitance depends on the gate voltage: 7.48 fF/µm² at 0.9 V, 6.5 at 0.6 V, only 2.2 at 0 V (measured C-V, tt). If V_cont drops below ~0.6 V, C1 collapses, the zero moves up and the PM drops. So the VCO fine-tune range must be centred so V_cont stays in 0.6–1.2 V — a requirement handed to steps 74/75.

</details>

<details><summary><b>Q13.</b> Why is C1 allowed to be a MOS cap but C2 is a MIM cap?</summary>

C1 is the large zero capacitor; a ±19 % variation of it moves the zero and changes PM by only a few degrees. C2 sets the pole and directly takes the charge packet each cycle; its linearity and its absolute value matter more, and it is small (2.2 pF), so a linear MIM costs little area.

</details>

<details><summary><b>Q14.</b> How is R1 realised and why that resistor type?</summary>

res_xhigh_po_2p85 (2.85 µm wide, 701 Ω/µm, 139 Ω heads), L = 15.5 µm → 11.0 kΩ. In the ngspice R/C corners it moves −12 % / +13 % (mostly temperature: tc ≈ −2.5e-3/°C), against −17 % / +20 % for res_high_po. It is compact, and wide enough for low mismatch and flicker. 2.85 µm is one of the five legal widths.

</details>

<details><summary><b>Q15.</b> How much does R1 add to the jitter, and what is the trade-off with β?</summary>

0.31 ps FFNC-off / 0.165 ps FFNC-on (total 2.12 / 1.36 ps). Doubling β halves R (−30 % R noise) but doubles C1 and C2 area; halving β halves area but R-jitter rises to 0.40 ps. Between β = 0.5 and 2 µA/rad the total changes by only 0.03 ps, so β is chosen for area and for a Gm small enough to keep its own noise and offset low.

</details>

<details><summary><b>Q16.</b> Why are the zero and pole at 0.84 MHz and 7.4 MHz?</summary>

They are placed about geometrically around f_c with spread b = tan(45° + lead/2) ≈ 2.9: f_z = f_c/b, f_p = f_c·b (here slightly asymmetric because the sampled-loop correction adds lead). The zero gives the phase lead at f_c; the pole filters the per-cycle charge-packet ripple and the reference spur.

</details>

<details><summary><b>Q17.</b> What is the closed-loop peaking and the 3 dB bandwidth?</summary>

2.5 dB peaking, f_3dB = 4.3 MHz. Peaking amplifies in-band (reference, SSPD, CP) noise near f_c; 2.5 dB is acceptable because those sources are ≥ 10 dB below the VCO contribution in the budget.

</details>

<details><summary><b>Q18.</b> How do you check the loop's lock behaviour at this stage?</summary>

With the same sampled model, cycle by cycle: after a 1 rad reference phase step the error overshoots 27 % and settles inside ±10 ps in 0.34 µs. The 5 µs lock-time spec is therefore set by FLL acquisition (step 81), not by the phase loop.

</details>

<details><summary><b>Q19.</b> Which parameters make up the loop-gain spread m?</summary>

m = (A·g_m·τ·K_VCO)/nominal. A (SSPD amplitude, buffer swing), g_m (bias, mobility), τ (pulser RC) and K_VCO (ring) all vary over PVT; their product can span about ×0.3 to ×2.4 before any trim.

</details>

<details><summary><b>Q20.</b> How robust is the loop to m and to the R/C corners?</summary>

At nominal R/C, PM ≥ 45° for m = 0.46–1.75. At the worst R/C combinations (R −12/+13 %, MIM −12/+13 %, MOS cap −19/+8 % over V_cont 0.6–1.2 V, ss/ff, −40/110 °C) the range narrows to m = 0.78–1.23. With a residual m of 0.75–1.3, PM stays 45.5–50° and f_c 1.8–3.3 MHz at every corner.

</details>

<details><summary><b>Q21.</b> What does icp_trim do and how is it built?</summary>

It changes the number of identical Gm unit cells: 00 → 3 units (nominal), 01 → 4, 10 → 2, 11 → 5, i.e. β ×1, ×1.33, ×0.67, ×1.67. It recentres the process part of m (≈ ×0.55 to ×2.0) to within about ±17 %, leaving room for voltage and temperature drift. Unit cells (rather than a trimmed resistor or pulse width) keep the layout regular and the Gm noise proportional to β.

</details>

<details><summary><b>Q22.</b> Why should code 00 be the nominal setting?</summary>

All controls power up at 0. With 00 = nominal the PLL locks with a sensible bandwidth before any calibration, which makes bring-up and the first measurements safe. The offset-binary decode costs three gates on VDDD.

</details>

<details><summary><b>Q23.</b> What happens if PM drops to 40°?</summary>

The loop is still stable, but peaking rises to about 4 dB, the overshoot grows, and in-band noise is amplified near f_c. The jitter model at PM 40° gives out2 pess 1.84 ps — still inside the 3.2 ps max — so 40° is the floor, 45° the target.

</details>

<details><summary><b>Q24.</b> Why is the continuous approximation good below f_c but wrong near f_REF/2?</summary>

Below f_c the loop sees many samples per cycle of any disturbance, so sampling is invisible. Near f_REF/2 the sampled gain folds and its phase falls faster (delay e^{−jωt_p} plus the hold): the sampled |L| flattens at −25 dB while the continuous one keeps falling. Gardner's rule (f_c < f_REF/10) keeps the difference small; here f_c/f_REF = 1/20.

</details>

<details><summary><b>Q25.</b> What is Gardner's stability limit and how far are we from it?</summary>

For a CP-PLL the sampled loop becomes unstable when the loop gain is so high that the per-cycle correction overshoots; roughly ω_c²·T² limits apply when f_c approaches f_REF/π. At f_c = f_REF/20 we are far below; m = 3 (f_c ≈ 6 MHz) still keeps PM above 30°.

</details>

<details><summary><b>Q26.</b> How does the FLL interact with this loop filter?</summary>

The FLL charge pump injects into the same filter while out of lock. With C_tot ≈ 19 pF a 10 µA FLL current slews V_cont at 0.5 V/µs; once the phase error is inside the dead zone the FLL CP is off and only the pulsed Gm acts. Step 81 sizes the FLL current against this C_tot.

</details>

<details><summary><b>Q27.</b> Why does the filter not use a third pole (4th-order loop)?</summary>

The reference spur from a pulsed-Gm SSPLL is small (no PFD/CP mismatch pulses; in lock the packets are near zero), and C2 with the 7.4 MHz pole already filters the packet ripple at 50 MHz by about 17 dB. An extra RC pole would cost phase margin and area for a spur that is not the limiting spec.

</details>

<details><summary><b>Q28.</b> What leakage matters in this filter?</summary>

Gate leakage of the thin-oxide MOS cap and junction leakage at V_cont. A leakage I_L is supplied by the loop as a static phase offset θ = I_L/β and a small per-cycle ripple I_L·T/C2. For nA-level leakage the offset is mrad and the ripple µV, but it is checked in the loop-filter testbench (step 100) at 110 °C.

</details>

<details><summary><b>Q29.</b> How does temperature move f_c?</summary>

R (xhigh_po) falls about 19 % from −40 to 110 °C (+11 % / −10 % around 27 °C); K_VCO and g_m change too. Since f_c ≈ β·K_VCO·R/2π above the zero, f_c moves with the product; the robustness plot covers this with the R and m axes.

</details>

<details><summary><b>Q30.</b> Why freeze the loop filter now, before the VCO and CP exist?</summary>

C1 and C2 are the largest analog blocks after the CCO and the decap; their area and shape set the floorplan of the CP + LF region. Freezing them (and absorbing K_VCO uncertainty in g_m) lets the floorplan and layout start early.

</details>

<details><summary><b>Q31.</b> What is in the design sheet that the block designers need?</summary>

β·K_VCO product, g_m per unit, τ and t_p limits, R1/C1/C2 with device names and sizes, V_cont window 0.6–1.2 V, icp_trim mapping, and the spread budget for m (process ×0.55–2.0, residual ×0.75–1.3).

</details>

<details><summary><b>Q32.</b> How would you verify the loop parameters in simulation?</summary>

Step 100: AC of the filter impedance at R/C corners (zero, pole). Step 103: open-loop gain from extracted block gains. Step 104: transient lock with a phase/frequency step, compared with the cycle model's 0.34 µs and 27 % overshoot.

</details>

<details><summary><b>Q33.</b> Why does the jitter barely depend on β at fixed f_c and PM?</summary>

Because the only β-dependent noise is R1's, and R1's contribution is small (0.2–0.4 ps against 2.1 ps total). The detector and CP noise referred to VCO phase do not depend on how β is split between g_m and τ, only on A, g_m and d through the budget formulas of step 72.

</details>

<details><summary><b>Q34.</b> Explain the open-loop gain shape in the Bode plot.</summary>

Below the zero: two integrators (VCO and C_tot) → −40 dB/dec and −180°. Between zero and pole: −20 dB/dec, phase rises to a maximum near f_c (the lead). Above the pole: −40 dB/dec again. The sampled version adds extra phase lag from the delay and sampling and flattens near f_REF/2.

</details>

<details><summary><b>Q35.</b> What would change if the reference were 45 or 55 MHz?</summary>

T changes, so the per-cycle charge changes but the average gain β = 2A·g_m·τ/T changes with 1/T: β rises 11 % at 55 MHz and falls 10 % at 45 MHz (m = 1.11 / 0.90) — inside the residual band. f_c/f_REF stays near 1/20.

</details>

<details><summary><b>Q36.</b> Why is the loop parameterised by β and not by an equivalent charge-pump current I_p?</summary>

In a PFD/CP loop the CP current per radian of reference phase is I_p/2π and the divider adds 1/N. In the SSPLL there is no divider and the detector gain is per radian of VCO phase, so β (A/rad of VCO phase) is the natural unit; I_p/(2πN) of a CP-PLL corresponds to β.

</details>

<a id='step-73-hv'></a>
### Step 73 redo — loop parameters for 800 MHz with 5 V devices (18 Nov 2026)

**What happens.** The loop keeps its shape: $f_c$ = 2.5 MHz, PM = 50°, zero and pole at 0.88 / 7.13 MHz. What changes
is the big capacitor. $C_1$ must now be a thick-oxide MOS capacitor (`nfet_g5v0d10v5`), which stores 2.6× less charge
per µm² (2.87 against 7.48 fF/µm²) and is flat only above 1.0 V.

**How it happens.** Only $\beta K_{VCO}/C$ and $R_1C_1$ set the loop. Scale $C_1, C_2$ and $\beta$ down by the same
factor and $R_1$ up, and the loop is unchanged. The price is the thermal noise of the larger $R_1$, which reaches the VCO
through the filter divider:

$$S_{\varphi,R_1}(f) = 4kTR_1\left|\frac{C_1}{C_1+C_2+j2\pi fR_1C_1C_2}\right|^2\frac{K_{VCO}^2}{f^2}\,\big|(1-G_c)(1-H)\big|^2$$

**Why it happens this way.** Inversion capacitance is $C_{ox} = \varepsilon_{ox}/t_{ox}$: the thicker oxide of the
5 V device lowers it, and its higher threshold moves the flat region up. The window moves to 1.0–2.3 V, centred at
VDDA/2. With 2 tiles ($C_1$ = 8.81 pF, $R_1$ = 20.6 kΩ) the extra $R_1$ noise costs 0.013 ps and the filter fits in
about 3 800 µm². Full write-up: `docs/design/step73_hv_loop.md`.

In [ ]:
import sys, json, csv; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget"); sys.path.insert(0, "../design/loop")
import sampled_loop as sl, pllmodel as pm, step72_hv_budget as hv
DH = json.load(open("../design/loop/data/step73_hv_design.json"))
fh = pm.freq_axis(n=700)

print(f"f_c {DH['fc']/1e6:.2f} MHz, PM {DH['pm']:.1f} deg (t_p {DH['tp']*1e9:.1f} ns), zero {DH['fz']/1e6:.3f} MHz, pole {DH['fp']/1e6:.2f} MHz")
print(f"beta {DH['beta']*1e9:.0f} nA/rad -> g_m {DH['gm']*1e6:.1f} uS = 3 units x {DH['gm']/3*1e6:.2f} uS (A {DH['A']} V/rad, duty {DH['duty']})")
print(f"C1 {DH['C1']*1e12:.2f} pF = {DH['ntile']} tiles x {DH['tile_pF']:.3f} pF; C2 {DH['C2']*1e12:.3f} pF; R1 {DH['R']/1e3:.2f} kOhm "
      f"(xhigh_po_2p85, L {DH['R_len']:.1f} um); area ~{DH['area_um2']:.0f} um2")
print(f"peaking {DH['peak_dB']:.2f} dB, f_3dB {DH['f3dB']/1e6:.2f} MHz, overshoot {DH['overshoot_pct']:.1f} %, settles to +-10 ps in {DH['settle_s']*1e6:.2f} us")
print("R1 noise (contribution / total, ps):", {k: (round(a, 3), round(b_, 3)) for k, (a, b_) in DH["noise"].items()})
print(f"residual m 0.75-1.3 at all R/C corners: PM {DH['residual']['0.75-1.3'][0]:.1f}-{DH['residual']['0.75-1.3'][1]:.1f} deg")

**Figure 73.6 — the thick-oxide capacitor.** C-V of the 5 V MOS capacitor at three corners against the 1.8 V device of the
first design. The shaded band is the new $V_{cont}$ window.

In [ ]:
import csv
cvh = list(csv.DictReader(open("../design/loop/data/moscap_hv_cv.csv")))
cvl = list(csv.DictReader(open("../design/loop/data/moscap_cv.csv")))
fig = go.Figure()
for (c, t), col, w in ((("tt", "27"), "#2a78d6", 3), (("ss", "-40"), "#7fb0ea", 1.5), (("ff", "110"), "#7fb0ea", 1.5)):
    rr = [r for r in cvh if r["corner"] == c and r["temp"] == t]
    fig.add_trace(go.Scatter(x=[float(r["vg"]) for r in rr], y=[float(r["c_mos_fF"])/100 for r in rr], mode="lines+markers",
                             name=f"nfet_g5v0d10v5 {c} {t} °C", line=dict(color=col, width=w)))
rr = [r for r in cvl if r["corner"] == "tt" and r["temp"] == "27"]
fig.add_trace(go.Scatter(x=[float(r["vg"]) for r in rr], y=[float(r["c_mos_fF"])/100 for r in rr], mode="lines",
                         name="nfet_01v8 (first design) tt", line=dict(color="#eb6834", dash="dash")))
fig.add_vrect(x0=1.0, x1=2.3, fillcolor="#e6e8eb", opacity=0.5, line_width=0)
fig.update_layout(height=420, xaxis_title="V_cont (V)", yaxis_title="capacitance density (fF/µm²)", yaxis_range=[0, 9])
show(fig)

**Figure 73.7 — how many C1 tiles?** The loop is the same for any tile count; only $R_1$ and the area change. Slide
the number of tiles and watch the $R_1$ noise and the totals. Two tiles is the knee: three cost 1 500 µm² more for
0.013 ps.

In [ ]:
import sys, json, csv; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget"); sys.path.insert(0, "../design/loop")
import sampled_loop as sl, pllmodel as pm, step72_hv_budget as hv
DH = json.load(open("../design/loop/data/step73_hv_design.json"))
fh = pm.freq_axis(n=700)

TR = {t["ntile"]: t for t in DH["trade"]}
def frame(n):
    t = TR[n]; tr = []; note = []
    for on, col in ((False, "#2a78d6"), (True, "#eb6834")):
        ff, S = hv.spectra("typ", on, f=fh, lf=(t["R"], t["C1"], t["C2"]))
        k = "loop-filter resistor R1"
        tr.append(dict(x=ff, y=10*np.log10(np.maximum(S[k], 1e-30)/2), name=f"R1 noise, FFNC {'on' if on else 'off'}", line=dict(color=col)))
    return tr, (f"{n} tile(s): C1 {t['C1']*1e12:.1f} pF, R1 {t['R']/1e3:.1f} kΩ, area {t['area']:.0f} µm² — R1 jitter "
                f"{t['rj_off']:.3f} / {t['rj_on']:.3f} ps, total {t['tot_off']:.3f} / {t['tot_on']:.3f} ps (off / on)")
def lay(fig):
    fig.update_xaxes(type="log", title="offset (Hz)"); fig.update_yaxes(title="R1 contribution (dBc/Hz)", range=[-160, -110])
show(sweep([1, 2, 3, 4, 5, 6], frame, "C1 tiles", fmt="{:d}", active=1, layout=lay))

### Interview questions — step 73 — 800 MHz redo (8)

<details><summary><b>Q1.</b> Why does the 5 V MOS capacitor give 2.6× less capacitance per area, and why is its flat region higher?</summary>

In inversion a MOS capacitor is the gate oxide: C_ox = ε_ox/t_ox. The 5 V devices have a gate oxide roughly 2.5–3× thicker than the 1.8 V devices, so C_ox falls to 2.87 fF/µm² against 7.48. The thicker oxide also raises the threshold voltage, so the channel forms later: the 5 V cap is flat (within 3 %) only above 1.0 V, while the 1.8 V cap was flat above 0.6 V. Below that the capacitance collapses towards the depletion value (1.3 fF/µm² at 0 V).

</details>

<details><summary><b>Q2.</b> C1 has to be smaller now. Why does lowering β keep the loop identical, and what does it cost?</summary>

f_c and PM depend only on β·K_VCO/C and on R·C: C_tot = b·β·K_VCO/ω_c² and R1 = b/(ω_c·C1). Scale C1 and C2 by k and β by k, and R1 by 1/k: the zero (1/R1C1) and the pole stay put and the gain curve is unchanged. The cost is R1's thermal noise, 4kTR1, which grows as 1/k. Here 2 tiles (8.81 pF, R1 = 20.6 kΩ) cost 0.013 ps more than 3 tiles and save 1 500 µm².

</details>

<details><summary><b>Q3.</b> Why does f_c come out at 2.5 MHz again at 800 MHz?</summary>

The same two specs pull in opposite directions: L(300 kHz) with FFNC off (now −114 dBc/Hz) needs a wide loop, f_c ≥ about 2.25 MHz; the FFNC jitter reduction (35 % typ) needs a narrow one, f_c ≤ about 2.55 MHz. The jitter in seconds did not change with the carrier (same ring power and FoM), so the crossing point did not move either.

</details>

<details><summary><b>Q4.</b> Why does the phase-step settling to ±10 ps take longer at 800 MHz although the loop is identical?</summary>

The loop dynamics in radians are the same (27.6 % overshoot, same time constants), but the window is in seconds. At 800 MHz, 10 ps is 2π·0.8 GHz·10 ps = 0.050 rad against 0.10 rad at 1.6 GHz, so the error has to decay twice as far: 0.40 µs instead of 0.34 µs.

</details>

<details><summary><b>Q5.</b> Why is the level shifter in the Gm pulse path not timing-critical for the loop, while the one on the sampling edge is?</summary>

A delay in the Gm pulse only delays the charge packet: it adds 360°·f_c·t_p of phase lag — 0.9° for 1 ns at 2.5 MHz. Its jitter does not matter either, because the packet's charge, not its timing, carries the phase information. The sampling edge is the phase measurement itself: any jitter on it is reference jitter, multiplied by N² at the output.

</details>

<details><summary><b>Q6.</b> Why centre the V_cont window at VDDA/2 (1.0–2.3 V)?</summary>

The lower edge is set by the MOS capacitor (flat above 1.0 V). The upper edge leaves headroom for the Gm output cascodes and the FLL charge pump on the 3.3 V supply. Centring the window gives the Gm the most symmetric compliance and makes the VCO fine range symmetric around the lock point after the coarse code is set.

</details>

<details><summary><b>Q7.</b> When would you stack MIM2 on MIM1 in the C1 tile, and what must you check first?</summary>

cap_mim_m3_2 (met4/capm2, top plate on met5) has the same 2.0 fF/µm² and fits above MIM1, raising a tile from 4.4 to 6.2 pF. Use it when area is tight. First check that the slot lets you use met5 (the harness may reserve it for power) and that the MIM2 rules (enclosure, spacing to MIM1 vias) fit the 35 µm tile.

</details>

<details><summary><b>Q8.</b> Why does the flatter thick-oxide C-V improve the phase-margin spread at the C-min corner?</summary>

C1 sets the zero. At the 1.8 V design the MOS share of C1 varied ×0.81–1.08 over its window and corners; the thick-oxide cap varies only ×0.93–1.06 over 1.0–2.3 V. A smaller C1 drop moves the zero up less, so the lead at f_c and the PM drop less: the residual band m = 0.75–1.3 keeps PM ≥ 46.5° at every R/C corner (45.5° before).

</details>

<a id='step-74'></a>
## Step 74 — VCO redesign: stages, LVT, sizing for frequency, K_VCO and phase noise

*Roadmap step 74: schematic and sizing table*

### What happens
The ring's frequency is set by how fast each stage charges its load: more CCO current means faster edges. vtune hangs a
little extra capacitance on every output for fine tuning. Every transistor's noise moves the edges a little. White
(thermal) noise gives phase noise falling at −20 dB/dec. Slow flicker (1/f) noise is up-converted into close-in phase
noise falling at −30 dB/dec, and below the $1/f^3$ corner it dominates.

### How it happens
A current-fed ring settles at the supply voltage $V_{ring}$ where its average current equals the CCO current, so
$f \approx I/(N C_{node} V_{ring})$, $K_I = \partial f/\partial I$, $K_{VCO} = \partial f/\partial v_{tune}$.
Phase noise follows the ISF $h(\tau) = d\phi/dq$:
$$\mathcal{L}_{th}(\Delta f) = \frac{2N\langle h^2 S_{th}\rangle}{2(2\pi\Delta f)^2},\qquad
\mathcal{L}_{fl}(\Delta f) = \frac{2N\langle h\sqrt{B}\rangle^2}{\Delta f\,2(2\pi\Delta f)^2}.$$

### Why it happens this way
Flicker only becomes phase noise through the **dc part** of $h$ (here 0.67 of its rms), and a device's flicker density
scales like $1/(WL)$. The smallest devices on an output node therefore dominate: the fine-tune switch and the coupling TG
nMOS. The main inverters contribute only 13 %. Enlarging just those two devices fixes the noise for 9 % more current;
scaling the whole ring would cost power one-for-one.

**Methods used:** M1 (transient frequency), M2 (ISF with the models' noise), M3 (noise tables), M4 (LVS), M5 (drawing
checks). What, why and how for each: `docs/methods.md`.

**Figure 74.1 — the ISF of one ring node and who causes the flicker.** Hover to compare the base cell and the step-74 cell.

In [ ]:
import json, base64
from IPython.display import Image, display
D = lambda n: json.load(open("../design/vco/data/" + n))
base, q3, ch = D("step74_base.json"), D("step74_q3.json"), D("step74_char_q3.json")

fig = make_subplots(rows=1, cols=2, subplot_titles=("ISF h(τ) of node p1 (rad/pC)", "flicker share (% of the base cell's total)"))
for d, c, n in ((base, "#2a78d6", "base cell"), (q3, "#eb6834", "step-74 cell")):
    h = np.array(d["h"]); ph = np.arange(len(h))/len(h)
    fig.add_trace(go.Scatter(x=np.r_[ph, 1], y=np.r_[h, h[0]]/1e12, name=f"{n} (corner {d['fc3']/1e6:.2f} MHz)",
                             line=dict(color=c)), row=1, col=1)
names = {"x1.xp2": "main pMOS", "x1.xn2": "main nMOS", "x1.xs2": "tune switch S", "x1.xt1n": "TG nMOS (own)",
         "x2.xt1n": "TG nMOS (next)", "x1.xt1p": "TG pMOS (own)", "x2.xt1p": "TG pMOS (next)"}
ref = 10**(base["Lfl"]/10)
for d, c in ((base, "#2a78d6"), (q3, "#eb6834")):
    fig.add_trace(go.Bar(y=list(names.values()), x=[d["shares"][k][1]*10**(d["Lfl"]/10)/ref*100 for k in names],
                         orientation="h", marker_color=c, showlegend=False), row=1, col=2)
fig.update_layout(height=430, barmode="group"); fig.update_xaxes(title_text="injection time / period", row=1, col=1)
show(fig)
print(f"base: Lth {base['Lth']:.2f}, Lfl {base['Lfl']:.2f}, corner {base['fc3']/1e6:.2f} MHz, I {base['I']*1e3:.3f} mA")
print(f"step-74: Lth {q3['Lth']:.2f}, Lfl {q3['Lfl']:.2f}, corner {q3['fc3']/1e6:.2f} MHz, I {q3['I']*1e3:.3f} mA")

**Figure 74.2 — coarse tuning range and ring-node voltage over corners.** Slider: process/temperature corner.

In [ ]:
import json, base64
from IPython.display import Image, display
D = lambda n: json.load(open("../design/vco/data/" + n))
base, q3, ch = D("step74_base.json"), D("step74_q3.json"), D("step74_char_q3.json")

labs = {"tt_27": "tt, 27 °C, 10 fF", "ss_110": "ss, 110 °C, 15 fF", "ff_-40": "ff, −40 °C, 5 fF"}
def frame(k):
    r = ch["f_vs_I"][k]; I = [x[0]*1e3 for x in r]
    return ([dict(x=I, y=[x[1]/1e9 for x in r], name="f (GHz)", line=dict(color="#2a78d6"), row=1, col=1),
             dict(x=I, y=[1.44]*len(I), name="1.44 GHz", line=dict(color="#59636e", dash="dot"), row=1, col=1),
             dict(x=I, y=[1.76]*len(I), name="1.76 GHz", line=dict(color="#59636e", dash="dash"), row=1, col=1),
             dict(x=I, y=[x[2] for x in r], name="ring node (V)", line=dict(color="#eb6834"), row=1, col=2),
             dict(x=I, y=[1.8]*len(I), name="1.8 V limit", line=dict(color="#59636e", dash="dot"), row=1, col=2)],
            labs[k])
def lay(fig):
    fig.update_xaxes(title_text="CCO feed current (mA)")
show(sweep(list(labs), frame, "corner", fmt="{}", active=0, rows=1, cols=2,
           subplot_titles=("frequency (GHz)", "ring supply node (V)"), layout=lay))

**Figure 74.3 — fine tuning.** $K_{VCO}$ is negative; below about 0.7 V the switch turns off and the curve flattens, so the lock window is 0.7–1.2 V.

In [ ]:
import json, base64
from IPython.display import Image, display
D = lambda n: json.load(open("../design/vco/data/" + n))
base, q3, ch = D("step74_base.json"), D("step74_q3.json"), D("step74_char_q3.json")

fig = go.Figure()
cols = {"tt_27": "#2a78d6", "ss_110": "#eb6834", "ff_-40": "#1baf7a"}
for k, c in cols.items():
    r = [x for x in ch["f_vs_vtune"][k] if x[1]]
    fig.add_trace(go.Scatter(x=[x[0] for x in r], y=[x[1]/1e9 for x in r], name=k.replace("_", ", ") + " °C", line=dict(color=c)))
fig.add_vrect(x0=0.7, x1=1.2, fillcolor="#e6e8eb", opacity=0.5, line_width=0, annotation_text="lock window")
fig.update_layout(height=420, xaxis_title="vtune (V)", yaxis_title="f (GHz) at the 1.6 GHz current", title="Fine tuning")
show(fig)

**The schematics** (xschem, hierarchical: cell → ring core → VCO; LVS-clean against the simulated netlist).

In [ ]:
from IPython.display import Image, display
for n in ("sspll_vco_dcell", "sspll_vco_core", "sspll_vco", "sspll_vco_acbuf"):
    print(n); display(Image(filename=f"../docs/img/step74_{n}.png", width=900))

### Sizing table and results
| Device | Model | W × nf | L | Role |
|---|---|---|---|---|
| XP1/XP2 | pfet_01v8 | 3.0 µm × 4 | 0.15 µm | main inverter pMOS |
| XN1/XN2 | nfet_01v8 | 3.0 µm × 2 | 0.15 µm | main inverter nMOS |
| XS1/XS2 | nfet_01v8 | 4.5 µm × 1 | **0.30 µm** | fine-tune switch |
| XC1/XC2 | nfet_01v8 | 4.5 µm × 1 | 0.50 µm | fine-tune MOS cap |
| XT1N/XT2N | nfet_01v8 | 0.84 µm × 1 | **0.30 µm** | coupling TG nMOS |
| XT1P/XT2P | pfet_01v8 | 0.63 µm × 1 | 0.15 µm | coupling TG pMOS |

1.352 mA for 1.6 GHz (tt), ring node 1.38 V; $\mathcal{L}$(1 MHz) −98.66 dBc/Hz thermal, $1/f^3$ corner 0.60 MHz;
$K_{VCO}$ −326 MHz/V (0.6–1.2 V); $K_I$ 0.84 GHz/mA; 1.44–1.76 GHz over corners with the ring node ≤ 1.65 V (10 fF/node);
start-up < 9 ns. System: 2.04 / 1.32 ps (FFNC off/on), 9.55 mW, FoM −227.8 dB. Full report: `docs/design/step74_vco.md`.

### Interview questions — step 74 (32)

<details><summary><b>Q1.</b> Why a ring oscillator and not an LC oscillator for this PLL?</summary>

Area and tuning range. An LC tank at 1.6 GHz needs a nH inductor (hundreds of µm on a side in SKY130's thick metal) and gives a narrow range; a ring is a few hundred µm², tunes over more than ±30 % and is easy to lay out. Its phase noise is ~20 dB worse, which is exactly why this design uses a sub-sampling detector and FFNC: they suppress and cancel ring noise.

</details>

<details><summary><b>Q2.</b> Why 5 stages and not 4?</summary>

A pseudo-differential ring of N stages is two single-ended rings of N inverters coupled by the cross connection. With N = 4 each half is a 4-inverter ring - a non-inverting loop that can latch (two stable states). The proposal's start-up simulations latched at tt, ss and ff. With N = 5 each half is an odd ring and cannot latch; the crossing at the end keeps the differential loop inverting.

</details>

<details><summary><b>Q3.</b> Why not 3 stages?</summary>

The step-6 study: 3 stages run at 2.4 GHz for N = 48 with similar efficiency (VCO FoM within 1 dB) but more absolute jitter at equal power, push the SSPD, NCB and first divider to 2.4–2.6 GHz, and would have changed the submitted proposal. 5 stages at 1.6 GHz were kept.

</details>

<details><summary><b>Q4.</b> What does the always-on transmission gate between the two halves do?</summary>

It ties inp to outp (and inm to outm) so the two single-ended rings run in anti-phase: a weak in-phase coupling that enforces the differential mode. Without it the halves could drift into the common mode. It also adds load and, as the ISF showed, noise - which is why its nMOS was redesigned.

</details>

<details><summary><b>Q5.</b> How is the frequency tuned?</summary>

Coarse: the CCO current into the ring's supply node (5-bit DAC, step 75) - more current, faster edges. Fine: vtune turns on switch S, which hangs a MOS capacitor on each output; more vtune means more capacitance and a lower frequency, so K_VCO is negative (−326 MHz/V average over 0.6–1.2 V). The loop's charge-pump polarity is chosen to match.

</details>

<details><summary><b>Q6.</b> Why is K_VCO negative and does it matter?</summary>

Raising vtune connects more capacitance, slowing the ring. A negative gain is fine as long as the sign of the detector or the charge pump is flipped so the loop has negative feedback. It must be documented, because a wrong sign makes the loop run away to a rail instead of locking.

</details>

<details><summary><b>Q7.</b> What did the ISF reveal about the original cell?</summary>

Thermal noise was dominated by the main pMOS (68 %), but 86 % of the flicker up-conversion came from two small devices: the fine-tune switch (42 %) and the coupling TG nMOS (44 %) - both minimum length (0.15 µm). The main inverter nMOS, normally blamed for flicker, contributed only 4 %.

</details>

<details><summary><b>Q8.</b> Why does a tiny switch dominate the flicker?</summary>

Flicker noise density scales roughly with 1/(W·L): a 0.15 µm device has a tiny gate area, so its 1/f current is large. The switch's on-resistance sets how much of the tuning capacitor the output sees; slow fluctuations of that resistance change the load - that is directly a frequency modulation, i.e. up-converted flicker (the dc part of the ISF).

</details>

<details><summary><b>Q9.</b> How was the 1/f³ corner reduced and what did it cost?</summary>

The switch and the TG nMOS were drawn at L = 0.3 µm (switch W 2.25 → 4.5 µm to keep W/L; TG nMOS 0.63 → 0.84 µm). Corner 1.06 → 0.60 MHz (spec ≤ 0.6). The bigger devices add capacitance: 10 % more current for 1.6 GHz (1.225 → 1.352 mA, corrected values with the PDK .spiceinit). L = 0.5 µm was tried first and cost 21 % of speed - rejected.

</details>

<details><summary><b>Q10.</b> Why must ngspice load the PDK .spiceinit, and what happened when it did not?</summary>

The sky130 model files are written for HSPICE; the PDK's spinit sets ngbehavior=hsa so ngspice reads them the HSPICE way. The first step-74 runs did not load it: multi-finger devices were read wrongly and the ring ran 4.6 % slow (1.437 instead of 1.352 mA for 1.6 GHz). Single-finger devices were unaffected, which is why the device noise tables did not change. Found when the stand-alone testbench decks disagreed with the design scripts; everything was re-run.

</details>

<details><summary><b>Q11.</b> Why not simply make the main inverters bigger to lower the noise?</summary>

Scaling the whole ring by k lowers all noise by 10·log k but costs k× power - the FoM does not change. The ISF showed the flicker came from two small devices; fixing those costs 9 % power for 2.6 dB lower close-in noise, far better than scaling.

</details>

<details><summary><b>Q12.</b> What is the 1/f³ corner and why does it matter here?</summary>

The offset where up-converted flicker equals thermal phase noise; below it L(f) falls at −30 dB/dec. With the loop at 2.5 MHz, a corner near 1 MHz puts flicker noise right where the loop and FFNC have the least suppression, raising L(300 kHz) FFNC-off above the −110 dBc/Hz spec.

</details>

<details><summary><b>Q13.</b> What does h_dc/h_rms (0.67) tell you?</summary>

How asymmetric the ISF is: the dc part converts slow (flicker) noise into phase drift. A perfectly symmetric waveform would have h_dc = 0 and no flicker up-conversion. 0.67 is high - the rising and falling edges and the load of the tuning branch are not symmetric - so the remaining handle is device area of the noisy devices.

</details>

<details><summary><b>Q14.</b> Why use LVT devices or not?</summary>

pfet_01v8_lvt has a minimum length of 0.35 µm - far too slow for a 1.6 GHz ring. nfet_01v8_lvt would lower the ring-node voltage for a given frequency (more headroom for the CCO), but the standard-Vt ring already runs at 1.45 V at tt and within the limit at ss/110 °C; LVT would add leakage and spread. Kept as a fallback, not used.

</details>

<details><summary><b>Q15.</b> Why is the ring fed by a current and not by a voltage?</summary>

A current-fed ring (CCO) has low supply sensitivity - the high output impedance of the current source isolates it from VDDA noise - and the current directly sets the frequency (K_I ≈ 1.2 GHz/mA). The ring-node voltage settles wherever the ring needs it to draw that current.

</details>

<details><summary><b>Q16.</b> Why must the ring-node voltage stay below 1.8 V?</summary>

The ring uses 1.8 V core devices (absolute maximum 1.95 V) and the CCO output transistor needs headroom from VDDA (2.97 V minimum): every 100 mV more on the ring node costs CCO headroom and raises its noise (step 72).

</details>

<details><summary><b>Q17.</b> Why does every ring node get a buffer, even unused ones?</summary>

Equal loading: each node then has the same capacitance, so all stages have the same delay and the ISF is the same at every node (the 2N-node assumption of the noise calculation holds). It also makes the layout regular - 10 identical buffer cells - and any tap can be used later.

</details>

<details><summary><b>Q18.</b> Why AC-coupled buffers?</summary>

The ring's DC level moves with the CCO current (1.1–1.7 V); a buffer biased by the ring's DC would change its trip point and duty cycle with frequency. The coupling cap removes the DC and the feedback resistor biases the first inverter at its own trip point, so the buffer output is centred at any ring voltage.

</details>

<details><summary><b>Q19.</b> How is the frequency measured in simulation and why that way?</summary>

Transient with an ideal current feed, a differential kick at t = 0, and .meas of 16–20 zero crossings of vop − vom after start-up has settled. A ring has no DC operating point that oscillates, so a small-signal analysis cannot give the frequency; the transient is the simulation equivalent of a frequency counter.

</details>

<details><summary><b>Q20.</b> How was phase noise computed without Spectre PNOISE?</summary>

With the ISF: the charge-injection response h(t) of one node at 12 phases, the bias of every device over a cycle and the PDK's own thermal and flicker noise tables for each device type and length. L = 2N·⟨h²S_th⟩/(2Δω²) for thermal; the flicker part uses ⟨h·√B⟩². It tells not only the total but which device causes it.

</details>

<details><summary><b>Q21.</b> How accurate is the ISF estimate?</summary>

The level is about 6 dB pessimistic against ngspice transient noise (step 7), because the ISF with cyclostationary white noise treats every injection as uncorrelated and the tables use the device's small-signal noise at each instant. The ranking of devices and the 1/f³ corner are reliable. Spectre PNOISE in the Cadence VM is the cross-check (step 95).

</details>

<details><summary><b>Q22.</b> Why use ideal R and C in the buffer for the long characterisation runs?</summary>

The PDK poly resistor is a behavioural (voltage-dependent) model and makes the transient about 5× slower. The ideal replacement uses the PDK values including the resistor's 3 fF end parasitics, and reproduces the PDK-buffer frequency to 0.001 %; the ISF runs and the LVS use the real PDK devices.

</details>

<details><summary><b>Q23.</b> What is K_I and how is it used?</summary>

df/dI of the current-fed ring (Hz/A). It converts the CCO's output current noise into frequency noise (step 72: i_n ≤ 7.2 pA/√Hz) and sets the DAC step size in Hz (step 75).

</details>

<details><summary><b>Q24.</b> How do you check the schematic is the simulated circuit?</summary>

xschem netlists the schematic (LVS format) and netgen compares it with the netlist generated for simulation: all four levels (cell, buffer, core, VCO) match uniquely. The same flow is used for layout-versus-schematic later.

</details>

<details><summary><b>Q25.</b> What layout decisions are already in the schematic?</summary>

Integer finger counts (pMOS 4 × 3 µm, nMOS 2 × 3 µm) so each cell is one abutting row; the two halves are mirror images; the 5 cells are identical; the TGs, switches and caps are drawn per half; 10 identical buffers; MIM bottom plate on the driven ring node, top plate on the high-impedance buffer gate.

</details>

<details><summary><b>Q26.</b> Why is the MIM bottom plate on the ring node?</summary>

The bottom plate (met3) has parasitic capacitance to the substrate. On the strongly driven ring node it only adds a little load; on the high-impedance buffer gate it would form a divider with the coupling cap and attenuate the signal.

</details>

<details><summary><b>Q27.</b> Why is the ring supply node decoupled with only 0.5 pF in simulation?</summary>

The ring is current-fed: a large decap on its supply node would just be charged by the CCO current (it sets the start-up time) and filters the ring's own current ripple. Its value is a layout choice (step 109); 0.5 pF keeps start-up fast.

</details>

<details><summary><b>Q28.</b> How does the frequency change from tt to ss/110 °C, and how is it handled?</summary>

Slower devices and higher temperature lower the frequency at a given current; the CCO DAC supplies more current. The check is that 1.76 GHz is reachable at ss/110 °C with the ring node below the limit, and 1.44 GHz at ff/−40 °C with the DAC's minimum current.

</details>

<details><summary><b>Q29.</b> What sets the start-up time?</summary>

The CCO charging the supply node and decap, then the growth of the differential mode from noise or mismatch. With 5 stages the differential mode is the only unstable mode, so it starts in a few ns at all corners.

</details>

<details><summary><b>Q30.</b> Why 10 fF extra capacitance per node in the simulations?</summary>

A pre-layout allowance for wiring; corners use 5 fF (ff/−40 °C, fast case) and 15 fF (ss/110 °C, slow case). Step 118 replaces it with the extracted values.

</details>

<details><summary><b>Q31.</b> What is the VCO's contribution to the total jitter after this step?</summary>

With the step-74 ring numbers in the step-72 model: 2.04 ps FFNC off, 1.32 ps FFNC on (typ), L(300 kHz) −110.6 / −120.0 dBc/Hz - all inside the typ specs. Power rises to 9.55 mW typ (spec 'about 9'), FoM −227.8 dB unchanged.

</details>

<details><summary><b>Q32.</b> If power had to come back to 9.0 mW, where would you take it?</summary>

The LDO load reserve (0.33 mW), the dummy-buffer size (8 of 10 buffers only provide equal loading - they can be smaller inverters with the same input capacitance), and the NCB bias. Not from the ring: its current buys the noise margin.

</details>

<a id='step-74-hv'></a>
### Step 74 redo — the ring VCO in 5 V devices, 800 MHz (19 Nov 2026)

**What happens.** The ring becomes a **3-stage** pseudo-differential ring of `nfet/pfet_g5v0d10v5`, still current-fed by
the CCO. At 0.994 mA it runs at 800 MHz (tt) with the ring node at 1.89 V; its thermal phase noise is −105.2 dBc/Hz at
1 MHz and its 1/f³ corner 0.59 MHz — both inside the step-72 targets — for 3.69 mW instead of the 4.86 mW budgeted.

**How it happens.** The same physics as at 1.6 GHz: the ring settles where its current equals the CCO current, and
each device's noise reaches the phase through the impulse sensitivity function:

$$f \approx \frac{I}{N C V_{ring}},\qquad P_{ring} = I V_{ring} \approx N C V_{ring}^2 f,\qquad
\mathcal{L}_{th,ring} = \frac{2N\langle h^2 S_{th}\rangle}{2(2\pi\Delta f)^2}\ \propto\ \frac{\gamma\,kT}{C V_{ring}^2}\cdot\frac{f^2}{\Delta f^2}$$

**Why it happens this way.** Thick-oxide devices are slow, so only three stages reach 880 MHz at the slow corner. But they
make less thermal noise per $g_m$ (γ ≈ 1.1 and 0.5 instead of 1.9 and 1.2) and the ring swings over a larger voltage, so
the noise target is met at less power. Flicker again comes from the smallest devices on a node — the transmission-gate
nMOS and the tuning switch — and is fixed by drawing them at L = 1 µm. The tuning switch's high threshold moves the
$V_{cont}$ window to 1.3–2.3 V. Full write-up: `docs/design/step74_hv_vco.md`.

**Figure 74.6 — the design study.** Each bar is one ISF run at 800 MHz (about 6 minutes each). Slide through the variants:
the title shows what changed, the current it needs, the thermal noise and the 1/f³ corner; the two curves show which
devices make the thermal and the flicker noise.

In [ ]:
import json, os
DV = "../design/vco/data"
ld = lambda n: json.load(open(os.path.join(DV, f"step74hv_{n}.json")))
CH = json.load(open(os.path.join(DV, "step74hv_char_hl6.json")))
SY = json.load(open(os.path.join(DV, "step74hv_system.json")))

VAR = [("s2", "feasibility ring ×2 + switched-cap tuning"), ("halftune", "half the switch and cap"),
       ("hl", "+ TG nMOS at L = 1 µm"), ("hl3", "+ switch at L = 1 µm"), ("hl4", "main nMOS ×1.5"),
       ("hl5", "main nMOS ×0.5"), ("hl6", "main nMOS 6 µm, all ×1.25 (chosen)")]
LAB = {"x1.xp2": "main pMOS", "x1.xn2": "main nMOS", "x1.xs2": "tune switch", "x1.xt1n": "TG nMOS own",
       "x1.xt1p": "TG pMOS own", "x2.xt1n": "TG nMOS next", "x2.xt1p": "TG pMOS next"}
def frame(i):
    n, lab = VAR[i]; d = ld(n); ks = list(LAB)
    return ([dict(x=[LAB[k] for k in ks], y=[100*d["shares"][k][0] for k in ks], name="thermal %", mode="lines+markers",
                  line=dict(color="#2a78d6"), marker=dict(size=9)),
             dict(x=[LAB[k] for k in ks], y=[100*d["shares"][k][1] for k in ks], name="flicker %", mode="lines+markers",
                  line=dict(color="#eb6834"), marker=dict(size=9))],
            f"{n}: {lab} — {d['I']*1e3:.3f} mA, thermal {d['Lth']:.2f} dBc/Hz (need −104.4), 1/f³ corner {d['fc3']/1e6:.2f} MHz (need 0.6)")
def lay(fig):
    fig.update_yaxes(title="share of the node's phase noise (%)", range=[0, 80])
show(sweep(list(range(len(VAR))), frame, "variant", fmt="{:d}", active=6, layout=lay))

**Figure 74.7 — why the $V_{cont}$ window moved.** Frequency against vtune at three corners. Below about 1.3 V the
tuning switch hardly conducts, so the curve is flat and the loop gain would collapse; the window is 1.3–2.3 V.

In [ ]:
import json, os
DV = "../design/vco/data"
ld = lambda n: json.load(open(os.path.join(DV, f"step74hv_{n}.json")))
CH = json.load(open(os.path.join(DV, "step74hv_char_hl6.json")))
SY = json.load(open(os.path.join(DV, "step74hv_system.json")))

fig = go.Figure()
for (k, v), col in zip(CH["f_vs_vtune"].items(), ["#2a78d6", "#eb6834", "#1baf7a"]):
    fig.add_trace(go.Scatter(x=[a for a, f, _ in v], y=[f/1e6 for a, f, _ in v], mode="lines+markers", name=k.replace("_", " ") + " °C",
                             line=dict(color=col)))
fig.add_vrect(x0=1.3, x1=2.3, fillcolor="#e6e8eb", opacity=0.5, line_width=0)
fig.update_layout(height=420, xaxis_title="vtune (V)", yaxis_title="f (MHz)", title=f"K_VCO over 1.3–2.3 V: " +
                  ", ".join(f"{k.replace('_', ' ')} {x/1e6:.0f} MHz/V" for k, x in SY["K"].items()))
show(fig)

In [ ]:
import json, os
DV = "../design/vco/data"
ld = lambda n: json.load(open(os.path.join(DV, f"step74hv_{n}.json")))
CH = json.load(open(os.path.join(DV, "step74hv_char_hl6.json")))
SY = json.load(open(os.path.join(DV, "step74hv_system.json")))

print("range end points (current, ring node):")
for k, (I, f, vr, vmax) in CH["range"].items():
    print(f"  {k:14s} {I*1e3:.3f} mA, ring node {vr:.2f} V")
print("system with this ring (step-72 HV model, step-73 filter):")
for k, v in SY["budget"].items():
    print(f"  {k:9s} jitter {v['jit']:.2f} ps (ring {v['ring']:.2f}), L(300 kHz) {v['L300k']:.1f} dBc/Hz")
print(f"power FFNC on {SY['P_on']:.2f} mW, off {SY['P_off']:.2f} mW; FoM {SY['FoM']:.1f} dB; Gm unit {SY['gm_unit']*1e6:.1f} uS")

### Interview questions — step 74 — 800 MHz redo (8)

<details><summary><b>Q1.</b> Why three stages now, when the 1.8 V design had to use five?</summary>

Two reasons. Speed: thick-oxide devices are about 5× slower, and a 5-stage ring would top out well below 880 MHz at ss / 110 °C; three stages reach 1.0 GHz there at 1.8 mA. Latch-up: each half of this pseudo-differential ring is vop → m1 → p2 → vop, three inversions, an odd loop that cannot latch; the 4-stage ring failed because its halves were even loops. Three is the smallest odd count that oscillates.

</details>

<details><summary><b>Q2.</b> The first 5 V ring (feasibility ring ×2) had a 1/f³ corner of 1.5 MHz. What did the ISF say, and what fixed it?</summary>

The ISF split the noise by device: the transmission-gate nMOS (own and next stage) made 72 % of the flicker and 36 % of the thermal noise, the tuning switch 33 % of the thermal. Drawing the TG nMOS at L = 1 µm (its flicker per µm falls 7×), halving the tuning branch and then drawing the switch at L = 1 µm with the same W/L brought the corner to 0.68 MHz; narrowing the main nMOS made the edges more symmetric and gave 0.59 MHz.

</details>

<details><summary><b>Q3.</b> Why does narrowing the main nMOS lower the flicker up-conversion?</summary>

Flicker noise becomes close-in phase noise only through the average (dc) part of the ISF, h_dc. h_dc is large when the rising and falling edges differ, because a slow, low-frequency current then moves one edge more than the other. The 5 V pMOS is much weaker than the nMOS per µm, so a narrower nMOS makes the edges more alike: h_dc/h_rms fell from 0.64 (N 8 µm) to 0.58 (N 4 µm). The cost is thermal noise and speed, so the final cell takes 6 µm and scales everything ×1.25.

</details>

<details><summary><b>Q4.</b> The thick-oxide ring meets the step-74 FoM at less power than the budget allowed. Why?</summary>

Thermal phase noise per milliwatt depends on γ and on the swing. The 5 V devices have γ ≈ 1.1 instead of 1.9, and the ring swings over 1.9 V instead of 1.4 V. At 0.994 mA (3.69 mW from VDDA through the CCO, against 4.86 mW budgeted) the thermal noise is −105.2 dBc/Hz at 1 MHz, 0.8 dB better than the −104.4 needed.

</details>

<details><summary><b>Q5.</b> Why was the V_cont window moved from 1.0–2.3 V to 1.3–2.3 V?</summary>

The fine tuning works by turning on switch S, which hangs a MOS cap on the node. S conducts only while the node is below vtune − V_T, and the thick-oxide V_T with body effect is about 1.1–1.2 V. Below vtune ≈ 1.3 V the switch barely conducts, f hardly moves (−65 MHz/V at tt, −18 at ss) and the loop gain would collapse. Over 1.3–2.3 V K_VCO is −173 MHz/V at tt (−110 ss, −250 ff), steady enough for the icp_trim to handle.

</details>

<details><summary><b>Q6.</b> The first start-up run said the ring did not oscillate. What was wrong?</summary>

The read-out, not the ring. At power-up all nodes start at 0 V and v(vop) − v(vom) chatters around 0 V at the simulator's numerical noise level; the measure counted those chatter crossings as cycles and reported 66 GHz with nanovolt swing. Reading instead when |vd| first passes 1 V, and f after 40 ns, shows start-up in 7–18 ns at every corner, with and without 2 % mismatch, in the correct (differential) mode.

</details>

<details><summary><b>Q7.</b> Why did the buffer's first inverter go from L = 1 µm back to L = 0.5 µm?</summary>

At L = 1 µm the self-biased inverter drew little current but was too slow: at ss / 110 °C / 880 MHz its output swung only 1.1 V. At L = 0.5 µm with 1.5 / 0.6 µm it swings 2.86 V of 2.97 V there, for 89 µA at tt. Two active buffers then take 0.59 mW, inside the 0.99 mW budget; the four dummies have no supply.

</details>

<details><summary><b>Q8.</b> Why are there buffers on all six nodes when only clkp and clkm are used?</summary>

Every node must see the same load, or the six stage delays differ, the waveform loses its symmetry and h_dc rises — more flicker up-conversion — and the duty cycle at clkp/clkm drifts. The four dummies are the same MIM + gate input with no supply connection, so they cost area but no power.

</details>

<a id='step-75'></a>
## Step 75 — Current-fed VCO supply (CCO) with PDK-legal resistors

*Roadmap step 75: schematic*

### What happens
The ring has no supply of its own: a pMOS current source from VDDA (3.3 V) pushes a current $I_{CCO}$ into its supply
node, and the ring settles at the voltage where it draws exactly that current. The current is the coarse frequency
knob, set by a 5-bit code. IBIAS (5 µA from the harness) is the only reference.

### How it happens
$$I_{CCO} = M\,I_u\,(N_{base} + \text{code}) = 25 \times 2.5\,\mu\text{A} \times (11 + \text{code}) = 0.69\ldots2.63\ \text{mA}$$
IBIAS → 2-unit nMOS diode → unit sinks (11 always on + binary 1…16) → 1-unit pMOS diode MREF → RC filter
($R_F$ = 10 MΩ, $C_F$ = 7.3 pF) → 25-unit pMOS MOUT → ring node. The filter makes the reference side invisible above
$f_F = 1/(2\pi R_F C_F) \approx 2$ kHz; what is left is MOUT's own noise plus the resistor's:
$$S_{i,out} \approx 4kT\gamma g_{m,out} + g_{m,out}^2\,\frac{4kTR_F}{1+(2\pi f R_F C_F)^2}
\;\xrightarrow{f \gg f_F}\; 4kT\gamma g_{m,out} + \frac{g_{m,out}^2\,4kT}{(2\pi f C_F)^2 R_F}$$
so a **larger** $R_F$ gives **less** in-band noise. Frequency noise is $S_f = K_I^2 S_i$, phase noise $S_\phi = S_f/f^2$.

### Why it happens this way
A current feed isolates the ring from VDDA (high output impedance), and current is what the ring's frequency is
proportional to. The ±20 % IBIAS tolerance sets the code range (3.4 : 1). The 1:25 ratio keeps the reference branch's
own supply current at 4 % of the ring current, which is what brings the worst-case power back under 15 mW. Above the
1.8 V rail the ring's thin-oxide devices would be over-stressed, so an active clamp sinks any excess current once the
ring node reaches VDDL.

**Methods used:** M10 (DC operating points and compliance), M11 (.noise of the output current, AC supply rejection),
M12 (clamp with the ring's measured I–V), M13 (level-shifter transient), M4/M5 (LVS, drawing checks). `docs/methods.md`.

**Figure 75.1 — coarse tuning.** Slider: IBIAS. Each point is one corner × VDDA; the band is the current the ring needs.

In [ ]:
import json
import numpy as np
from IPython.display import Image, display
D = lambda n: json.load(open("../design/cco/data/" + n))
s, nz = D("step75_sim.json"), D("step75_noise.json")

dc = s["dc"]
def frame(ib):
    tr = []
    for code, c in ((0, "#1baf7a"), (10, "#2a78d6"), (31, "#eb6834")):
        r = [x for x in dc[:90] if x[3] == ib*1e-6 and x[4] == code]
        tr.append(dict(x=[f"{x[0]}/{x[1]}°C/{x[2]}V" for x in r], y=[-x[5]*1e3 for x in r], mode="markers",
                       name=f"code {code}", marker=dict(color=c, size=8)))
    tr += [dict(x=[f"{x[0]}/{x[1]}°C/{x[2]}V" for x in dc[:90] if x[3] == ib*1e-6 and x[4] == 0], y=[0.97]*10,
                name="code 0 must be ≤ 0.97 mA (ff, −40 °C)", line=dict(color="#1baf7a", dash="dot")),
           dict(x=[f"{x[0]}/{x[1]}°C/{x[2]}V" for x in dc[:90] if x[3] == ib*1e-6 and x[4] == 0], y=[2.02]*10,
                name="code 31 must be ≥ 2.02 mA (ss, 110 °C)", line=dict(color="#eb6834", dash="dot"))]
    return tr, f"IBIAS {ib} µA"
def lay(fig):
    fig.update_yaxes(title_text="I into ring node (mA)")
show(sweep([4, 5, 6], frame, "IBIAS (µA)", fmt="{}", active=1, layout=lay))

**Figure 75.2 — compliance and the clamp.** Left: current vs ring-node voltage at the least headroom; right: the ring node at ss/110 °C (where the ring needs the most voltage) for every code.

In [ ]:
import json
import numpy as np
from IPython.display import Image, display
D = lambda n: json.load(open("../design/cco/data/" + n))
s, nz = D("step75_sim.json"), D("step75_noise.json")

fig = make_subplots(rows=1, cols=2, subplot_titles=("I (mA) vs ring node (V): ss, 110 °C, VDDA 2.97 V, code 31",
                                                    "ring node (V) vs code: ss, 110 °C, VDDA 3.63 V"))
cp = np.array(s["compliance"])
fig.add_trace(go.Scatter(x=cp[:, 0], y=-cp[:, 1]*1e3, name="I out", line=dict(color="#2a78d6")), row=1, col=1)
for ib, c in ((4e-6, "#eb6834"), (6e-6, "#eda100")):
    r = [x for x in s["clamp"] if x[0] == ib]
    fig.add_trace(go.Scatter(x=[x[1] for x in r], y=[x[2] for x in r], name=f"IBIAS {ib*1e6:.0f} µA",
                             line=dict(color=c)), row=1, col=2)
fig.add_hline(y=1.8, line_dash="dot", line_color="#e87ba4", row=1, col=2)
fig.update_yaxes(range=[-0.6, 2.7], row=1, col=1); fig.update_layout(height=420)
show(fig)

**Figure 75.3 — output-current noise and supply rejection** (complete circuit with PDK resistors and capacitors).

In [ ]:
import json
import numpy as np
from IPython.display import Image, display
D = lambda n: json.load(open("../design/cco/data/" + n))
s, nz = D("step75_sim.json"), D("step75_noise.json")

fig = make_subplots(rows=1, cols=2, subplot_titles=("output noise (pA/√Hz)", "|ΔI/ΔVDDA| (µA/mV)"))
fig.add_trace(go.Scatter(x=nz["f"], y=np.array(nz["n"])*1e12, name="CCO noise", line=dict(color="#2a78d6")), row=1, col=1)
fig.add_hline(y=10.3, line_dash="dot", line_color="#eb6834", row=1, col=1)
ps = np.array(s["psr"])
fig.add_trace(go.Scatter(x=ps[:, 0], y=ps[:, 1]*1e3, name="supply → current", line=dict(color="#1baf7a")), row=1, col=2)
fig.update_xaxes(type="log"); fig.update_yaxes(type="log", row=1, col=1); fig.update_layout(height=420)
show(fig)
print("jitter contribution of the CCO: %.2f ps (FFNC off) / %.2f ps (on); system total %.2f / %.2f ps; L(300 kHz) %.1f / %.1f dBc/Hz"
      % (nz["off"][0], nz["on"][0], nz["off"][1], nz["on"][1], nz["off"][2], nz["on"][2]))

**The schematics** (CCO top → DAC, filter resistor, level shifter; LVS-clean against the simulated netlist).

In [ ]:
from IPython.display import Image, display
for n in ("cco", "cco_dac", "cco_rf", "ls_1v8_3v3"):
    print("sspll_" + n); display(Image(filename=f"../docs/img/step75_sch_{n}.png", width=900))

### Sizing table and results
| Device | Model | W × nf × m | L | Role |
|---|---|---|---|---|
| XNB | nfet_g5v0d10v5 | 1 × 1 × 2 | 4 µm | IBIAS input diode (2 units) |
| XU0…XU5 | nfet_g5v0d10v5 | 1 × 1 × (11, 1, 2, 4, 8, 16) | 4 µm | DAC unit sinks, 2.5 µA each |
| XS0…XS5 | nfet_g5v0d10v5 | 1 × 1 × same | 0.5 µm | DAC switches (en, c0…c4) |
| XMREF | pfet_g5v0d10v5 | 40 (2 × 20) × 1 | 2 µm | reference diode (1 unit) |
| XMOUT | pfet_g5v0d10v5 | 40 (2 × 20) × 25 | 2 µm | output mirror (25 units) |
| xrf (20 segments) | res_xhigh_po_0p35 | 0.35 | 87.48 µm each | $R_F$ = 10 MΩ |
| XCF | cap_mim_m3_1 | 30 × 30 × 4 | — | $C_F$ = 7.3 pF (over MOUT) |
| XPREP / XUREP | pfet_01v8 1/0.15 / nfet_g5v0 1/4 | | | clamp replica (sets vcl) |
| XPSEN / XRS | pfet_01v8 1/0.15 / xhigh_po_0p35 56 µm | | | clamp sense, 320 kΩ |
| XNSH | nfet_01v8 | 20 (4 × 5) | 0.15 µm | clamp shunt |
| XTGN / XTGP / XRINJ | g5v0 2/0.5 / xhigh_po_0p35 3.39 µm | | | VTEST injection, 20 kΩ |

Every device passes `tools/pdk_limits.py`. tt code 0/10/31: 0.71 / 1.34 / 2.67 mA (after the 15 Nov correction, NBASE 11); over 5 corners × VDDA 2.97–3.63 V
× IBIAS 4–6 µA: code 31 ≥ 2.20 mA (needs 2.02), code 0 ≤ 1.02 mA (needs 0.97 — the 50 µA excess at ff/−40 °C/IBIAS
6 µA is ≈ 60 MHz, inside the ±75 MHz fine-tune range). Ring node ≤ 1.793 V in the worst case. Output noise 8.9 pA/√Hz at
1 MHz (needs ≤ 10.3); CCO jitter 0.70 / 0.44 ps, system 1.96 / 1.27 ps, L(300 kHz) −110.9 / −120.0 dBc/Hz.
Supply → current ≤ 0.25 µA/mV. Level shifter: full 3.63 V swing, rise 1.4–1.6 ns. Full report: `docs/design/step75_cco.md`.

### Interview questions — step 75 (28)

<details><summary><b>Q1.</b> What is the CCO in this design and why is the ring fed by a current?</summary>

A programmable current source from VDDA (3.3 V) into the ring's supply node. A current-fed ring settles at the voltage where it draws exactly that current, so the current sets the frequency (K_I ≈ 0.85 GHz/mA). The current source's high output impedance isolates the ring from VDDA noise, and the DAC gives the coarse tuning.

</details>

<details><summary><b>Q2.</b> Draw the CCO from IBIAS to the ring node.</summary>

IBIAS (5 µA) → 2-unit nMOS diode → unit sinks of 2.5 µA (11 base + binary 1, 2, 4, 8, 16, switched by cco_code) → pMOS diode MREF (1 unit) → 10 MΩ / 7.3 pF RC filter → MOUT (25 units) → ring node. I_out = 25 × 2.5 µA × (11 + code) = 62.5 µA × (11 + code): 0.69–2.63 mA.

</details>

<details><summary><b>Q3.</b> Why a 1:25 output mirror and not 1:8?</summary>

The reference branch also draws its current from VDDA: with 1:8 that is +12.5 % of the ring current, with 1:25 only +4 %. At the worst corner (2.0–2.3 mA at 3.63 V) that saves 0.6–0.7 mW, which brings the worst-case power back under 15 mW. The price: reference-side noise is multiplied by 25 — which the RC filter removes.

</details>

<details><summary><b>Q4.</b> Why is the DAC range 0.69–2.63 mA when the ring only needs 0.89–1.93 mA?</summary>

IBIAS from the harness may be 4.0–6.0 µA (±20 %), and the DAC current scales with it. At IBIAS = 6 µA, code 0 must still be low enough for 1.44 GHz at ff/−40 °C (0.89 mA); at 4 µA, code 31 must still reach 1.76 GHz at ss/110 °C (1.93 mA at 10 fF/node). The ±20 % costs a 3.8:1 code range (11 base + 31 units). The base count was moved from 13 to 11 when the corrected ring simulation lowered the ff requirement.

</details>

<details><summary><b>Q5.</b> How fine is the coarse tuning and why is it fine enough?</summary>

One LSB is 62.5 µA, i.e. about 53 MHz at tt (K_I 0.85 GHz/mA). The fine-tune range over the 0.75–1.2 V lock window is about 150 MHz, roughly 3 LSBs, so after the coarse code is chosen the loop can always reach lock with margin for temperature drift.

</details>

<details><summary><b>Q6.</b> Why the RC filter between the two mirror gates?</summary>

Without it, the noise of the reference side (MREF, the DAC units, the IBIAS input) appears at the output multiplied by the mirror ratio (×25 in current, ×625 in power). The RC low-pass with a ~2 kHz corner removes it everywhere in the 10 kHz–40 MHz band, so only MOUT's own noise and the filter resistor's noise remain.

</details>

<details><summary><b>Q7.</b> Why is the filter resistor 10 MΩ and not 300 kΩ?</summary>

Above the RC corner the resistor's own noise at the gate falls as √(4kT/R)/(2πCf): a larger R with the same C gives LESS noise in band. With 300 kΩ the output noise was 21 pA/√Hz at 1 MHz and L(300 kHz) FFNC-off −104.4 dBc/Hz; with 10 MΩ it is 8.9 pA/√Hz and −110.9 dBc/Hz.

</details>

<details><summary><b>Q8.</b> How is a 10 MΩ resistor made legal and layout-friendly?</summary>

20 series segments of res_xhigh_po_0p35 (the highest sheet resistance, 0.35 µm is a legal width), each 87.48 µm long (≈ 500 kΩ including heads), drawn as a serpentine. Every segment is inside the PDK limits and the array folds into a compact block.

</details>

<details><summary><b>Q9.</b> Where does the 7.3 pF filter capacitor go?</summary>

Four 30 × 30 µm MIM tiles, placed over the output-transistor array (the MIM layers are above the active devices), so they cost no extra area. Top plate on the high-impedance gate node, bottom plate on VDDA. MOUT's own gate capacitance (several pF) adds to it.

</details>

<details><summary><b>Q10.</b> Why is the filter capacitor tied to VDDA and not to ground?</summary>

The output pMOS current depends on its source-gate voltage. With the capacitor to VDDA, a VDDA step moves the gate with the source, so Vsg — and the current — stays constant: supply rejection. A capacitor to ground would hold the gate still and turn every VDDA ripple into a current ripple.

</details>

<details><summary><b>Q11.</b> What sets the CCO output noise floor?</summary>

MOUT's channel noise, i_n² = 4kTγ·g_m. With 25 units of 40/2 µm and ~1.45 mA, g_m is a few mS; the result is 8.9 pA/√Hz at 1 MHz (requirement ≤ 10.3). Lower g_m (larger overdrive) would help but costs headroom: VDDA_min − V_ring_max = 2.97 − 1.8 = 1.17 V.

</details>

<details><summary><b>Q12.</b> Why thick-oxide (g5v0d10v5) devices in the CCO?</summary>

Every device connected to VDDA can see up to 3.63 V (3.96 V absolute) between terminals; thin-oxide devices are rated for 1.8 V. The project rule: 3.3 V devices wherever a node touches VDDA or a pad.

</details>

<details><summary><b>Q13.</b> Why L = 2 µm for the mirror and 4 µm for the DAC units?</summary>

Long channels lower flicker noise (area), raise output impedance (better DAC accuracy and supply rejection) and improve matching. The output still saturates: Vov ≈ 0.5–0.7 V against 1.17 V of headroom.

</details>

<details><summary><b>Q14.</b> What does the clamp do and why is it needed?</summary>

At ss/110 °C with IBIAS +20 % and a high code, the CCO could push the ring node to ~2.0 V — above the 1.8 V rating of the ring's core devices and above the proposal's absolute limit. The clamp sinks the excess current once the node reaches the 1.8 V rail (VDDL).

</details>

<details><summary><b>Q15.</b> Why did a simple one-transistor clamp not work?</summary>

A pMOS from the ring node to ground, biased by a replica, conducts by Vsg: to sink ~1 mA at 1.8 V it already steals 0.4–0.85 mA at 1.7 V, which is exactly where the ring must run at the top of its range. Its I-V is not sharp enough.

</details>

<details><summary><b>Q16.</b> How does the two-stage active clamp get a sharp knee?</summary>

A small sense pMOS (gate at the replica voltage) conducts exponentially (weak inversion) as the ring node approaches VDDL; its current into 320 kΩ drives the gate of an nMOS shunt. The shunt only turns on when that voltage reaches its threshold, adding gain: < 10 µA at 1.725 V, 0.2–1.5 mA at 1.8 V. With the ring's I-V at ss/110 °C, code 31 and IBIAS 6 µA the node stays at 1.793 V.

</details>

<details><summary><b>Q17.</b> Why is the clamp referenced to VDDL and a replica pMOS?</summary>

The replica diode from the 1.8 V rail sets the sense gate one |Vgs| below VDDL, so the sense device turns on when the ring node reaches VDDL regardless of the pMOS threshold at that corner (it tracks process and temperature). VDDL is the regulated rail whose voltage is, by definition, safe for 1.8 V devices.

</details>

<details><summary><b>Q18.</b> How is the CCO disabled (pll_en = 0)?</summary>

A thick-oxide pMOS pulls the output gate to VDDA (MOUT off) and a thick-oxide nMOS shorts the IBIAS input to ground (all DAC units and the clamp replica off). The base units' switch is also driven by pll_en. All controls default to 0, so the block powers up off.

</details>

<details><summary><b>Q19.</b> What is VTEST for and how is it injected?</summary>

The test plan uses a tone on ANA0 (VTEST) to calibrate the cancellation gain (50 kHz) and to measure supply sensitivity (1 mV pp at 100 kHz). A thick-oxide transmission gate (vtest_en) and a 20 kΩ poly resistor inject (V_test − V_ring)/20 kΩ into the ring node: 1 mV → 50 nA → ~42 kHz frequency deviation, a clearly measurable spur.

</details>

<details><summary><b>Q20.</b> Why level shifters, and how does yours work?</summary>

cco_code, pll_en and vtest_en come from the 1.8 V digital domain but drive 3.3 V switches. A thin-oxide inverter on VDDD makes in/inb; thick-oxide nMOS pull-downs flip a cross-coupled thick-oxide pMOS latch on VDDA; output inverters give full 3.3 V out/outb. Pull-downs (W 3 µm) are sized to win against the pMOS loads (W 1 µm) even at VDDD 1.71 V.

</details>

<details><summary><b>Q21.</b> How was the CCO verified?</summary>

DC operating points (current vs code over 5 corners × VDDA × IBIAS), a compliance sweep, the clamp with the ring's measured I-V, an AC supply-rejection analysis, .noise of the complete circuit with PDK resistors and capacitors (converted to jitter through the loop model), and a level-shifter transient at the hardest supplies. Methods M10–M13.

</details>

<details><summary><b>Q22.</b> How do you convert CCO current noise to phase noise?</summary>

Frequency noise S_f = K_I²·S_i (Hz²/Hz); phase noise S_φ = S_f/f²; L = S_φ/2. In the PLL it is shaped like VCO noise: |(1 − G_c)(1 − H)|², so low-frequency CCO noise is suppressed by the loop and FFNC.

</details>

<details><summary><b>Q23.</b> Why did the CCO beat its step-72 allocation?</summary>

The allocation assumed 7.2 pA/√Hz with a 0.3 MHz flicker corner; the measured K_I (0.85 instead of 1.22 GHz/mA) relaxes the current-noise requirement by 1.44×, and the long-channel devices plus the filter keep the spectrum low. Result: 0.70 ps (FFNC off) / 0.44 ps (on) instead of 0.90 / 0.56 ps.

</details>

<details><summary><b>Q24.</b> What would you check in layout for this block?</summary>

Common-centroid placement of MREF among the 25 MOUT units, dummies at the array ends, the MIM tiles over the array, the serpentine resistor away from the ring and shielded, the gate node (vgo) kept short and guarded (it is a 10 MΩ high-impedance node: leakage and coupling matter), thick-oxide devices in their own well/guard ring.

</details>

<details><summary><b>Q25.</b> What leakage could upset a 10 MΩ / 7 pF filter node?</summary>

The gate node draws no DC current, but junction leakage of the enable pMOS drain and the resistor body leakage flow through 10 MΩ: 1 nA gives 10 mV of offset between vgr and vgo, i.e. a few % current error — a static offset the loop absorbs, but it is verified at 110 °C (leakage doubles every ~10 °C).

</details>

<details><summary><b>Q26.</b> How does VDDA noise reach the ring?</summary>

Through the finite output impedance of MOUT and through the gate coupling. The AC analysis gives ΔI/ΔVDDA of about 0.25 mA/V up to 1 MHz, i.e. 1 mV of VDDA ripple → 0.25 µA → ~0.2 MHz deviation — further reduced by the loop and FFNC below 2.5–6 MHz.

</details>

<details><summary><b>Q27.</b> What is the start-up behaviour?</summary>

pll_en rises, the level shifter releases the enable devices, IBIAS flows into the DAC and the reference diode, the filter charges the output gate with τ = RC ≈ 70 µs — the current ramps up over ~0.2 ms. The FLL waits for it (step 81).

</details>

<details><summary><b>Q28.</b> Could the 70 µs RC time constant be a problem for lock time?</summary>

Only at power-up: once the CCO current has settled, the coarse code is static. The 5 µs lock-time spec is measured 'from pll_en with coarse code set' — but the code must be set before the CCO is enabled, and the slow filter is precisely why the test plan sets cco_code first.

</details>

<a id='step-76'></a>
## Step 76 — REF buffer (REF_IN receiver): low-noise edge

*Roadmap step 76: schematic*

### What happens
The crystal oscillator's waveform arrives on the REF_IN pad: anything from a 0.4 V pp sine to a 0–3.3 V square wave.
The receiver AC-couples it, amplifies it around its own switching point and delivers a 1.8 V CMOS edge, `refclk`, to the
clock generator. Every bit of noise at the moment of the crossing moves the edge in time; inside the loop bandwidth
that timing error reaches the output multiplied by $N = 32$ in phase ($N^2$ = +30 dB in power).

### How it happens
Noise $v_n$ at a crossing of slope $s$ shifts the edge by $v_n/s$:
$$\sigma_t = \frac{\sigma_v}{s},\qquad s = \pi f_{ref} V_{pp}\ \text{(sine)},\qquad
\mathcal{L}_{floor} = \frac{(2\pi f_{ref}\,\sigma_t)^2}{f_{ref}} .$$
The receiver's own white noise $S_w$ is integrated over an effective bandwidth $B_{eff}$ (about S1's gain-bandwidth):
$\sigma_v^2 \approx S_w B_{eff}$. The first stage S1 is a current-reuse inverter (XN1/XP1, 120 µm each) fed by a 60 µA
header current source; $R_B$ = 5 MΩ from its output to its input biases it exactly at its trip point, $C_{AC}$ = 3.6 pF
couples the signal in, $C_H$ = 1.8 pF makes the header node an AC ground so both devices add $g_m$.

### Why it happens this way
Jitter per edge is set by noise per unit of input slope, so a steep input is the cheapest fix: a full-swing input is
3.3× steeper than 1 V pp and gives 10 dB less phase noise. On chip, lower noise costs $g_m$, i.e. current: moderate
inversion behind a current source gives the most $g_m$ per µA, and $C_H$ lets the pMOS contribute too.

**Methods used:** M10 (bias point), M14 (noise at the crossing point), M15 (transient-noise jitter), M16 (large-signal
transient, supply sensitivity, start-up), M9 (system budget), M4/M5 (LVS, drawing checks). `docs/methods.md`.

**Figure 76.1 — receiver phase noise vs input amplitude.** Slider: amplitude of the 50 MHz sine at the pad.

In [ ]:
import json
import numpy as np
from IPython.display import Image, display
D = lambda n: json.load(open("../design/refbuf/data/" + n))
nz, sim, tn = D("step76_noise.json"), D("step76_sim.json"), D("step76_tnoise.json")

fl = nz["fL"]
def frame(v):
    k = str(v)
    return ([dict(x=fl, y=nz["L"][k], name=f"receiver, {k} V pp", line=dict(color="#2a78d6")),
             dict(x=fl, y=[-155]*len(fl), name="step-72 allocation", line=dict(color="#e87ba4", dash="dot"))],
            f"{k} V pp: σ = {nz['sigma'][k]['abs']*1e12:.2f} ps per edge — system {nz['sys'][k]['false']['total_ps']:.2f} / "
            f"{nz['sys'][k]['true']['total_ps']:.2f} ps, L(300 kHz) {nz['sys'][k]['false']['L300k']:.1f} / {nz['sys'][k]['true']['L300k']:.1f} dBc/Hz")
def lay(fig):
    fig.update_xaxes(type="log", title_text="offset (Hz)"); fig.update_yaxes(title_text="L at 50 MHz (dBc/Hz)", range=[-170, -135])
show(sweep([0.4, 1.0, 3.3], frame, "V_pp", fmt="{}", active=2, layout=lay))

**Figure 76.2 — transient-noise edge times (method M15).** Each point is one `refclk` rising edge; the spread is the per-edge jitter.

In [ ]:
import json
import numpy as np
from IPython.display import Image, display
D = lambda n: json.load(open("../design/refbuf/data/" + n))
nz, sim, tn = D("step76_noise.json"), D("step76_sim.json"), D("step76_tnoise.json")

fig = go.Figure()
for r, c in zip([r for r in tn["runs"] if r["Sw"] > 0], ("#2a78d6", "#1baf7a", "#eb6834")):
    x = np.array(r["dev"]); k = np.arange(len(x)); x = x - np.polyval(np.polyfit(k, x, 4), k)
    fig.add_trace(go.Scatter(y=x*1e12, mode="lines", name=f"{r['vpp']} V pp (σ {x.std()*1e12:.2f} ps)", line=dict(color=c, width=1)))
fig.update_layout(height=400, xaxis_title="edge number", yaxis_title="deviation from the ideal grid (ps)")
show(fig)

**Figure 76.3 — delay, duty and supply current over corners and amplitudes** (drawn circuit with PDK passives). Hover for the condition.

In [ ]:
import json
import numpy as np
from IPython.display import Image, display
D = lambda n: json.load(open("../design/refbuf/data/" + n))
nz, sim, tn = D("step76_noise.json"), D("step76_sim.json"), D("step76_tnoise.json")

fig = make_subplots(rows=1, cols=3, subplot_titles=("delay pad → refclk (ns)", "duty (%)", "power (mW)"))
lab = [f"{r['corner']}/{r['temp']}°C/{r['vdda']} V/{r['vpp']} V pp {r['wave']}/{r['f']/1e6:.0f} MHz/VDDL {r['vddl']}" for r in sim["tran"]]
for i, (k, sc) in enumerate((("delay", 1e9), ("duty", 100), ("power", 1e3))):
    fig.add_trace(go.Scatter(x=list(range(len(lab))), y=[r[k]*sc for r in sim["tran"]], mode="markers", text=lab,
                             marker=dict(color="#2a78d6", size=7), showlegend=False), row=1, col=i + 1)
fig.add_hline(y=45, line_dash="dot", line_color="#eb6834", row=1, col=2)
fig.update_xaxes(title_text="condition #"); fig.update_layout(height=400)
show(fig)

**The schematics** (receiver top → S1 cell, R_B cell, level shifter from step 75; LVS-clean against the simulated netlist).

In [ ]:
from IPython.display import Image, display
for n in ("refbuf", "refbuf_s1", "refbuf_rb"):
    print("sspll_" + n); display(Image(filename=f"../docs/img/step76_sch_{n}.png", width=900))

### Sizing table and results
| Device | Model | W per finger × nf × m | L | Role |
|---|---|---|---|---|
| XRPU / XRPD | res_xhigh_po_0p35 | 0.35 | 87.48 µm | pad divider, 500 kΩ each |
| XCAC | cap_mim_m3_1 | 30 × 30 µm × 2 | — | $C_{AC}$ = 3.6 pF (bottom plate on the pad) |
| xrb (10 segments) | res_xhigh_po_0p35 | 0.35 | 87.48 µm | $R_B$ = 5 MΩ |
| XPD / XPH | pfet_g5v0d10v5 | 2 × 1 × (1 / 24) | 4 µm | bias diode / header (60 µA) |
| XCH | cap_mim_m3_1 | 30 × 30 µm | — | $C_H$ = 1.8 pF |
| XP1 / XN1 | p/nfet_g5v0d10v5 | 10 × 12 | 1 µm | S1, moderate inversion; XP1 body = source |
| XPEN / XNS1 | p/nfet_g5v0d10v5 | 2 × 1 | 0.5 µm | enable |
| XP2 / XN2 | p/nfet_g5v0d10v5 | 2 / 1 | 0.5 µm | S2 (VDDA) |
| XPL / XNL | p/nfet_g5v0d10v5 | 2 / 1 | 0.5 µm | 3.3 → 1.8 V inverter on VDDL |
| XPB, XPB2 / XNB, XNB2 | p/nfet_01v8 | 4 / 1 | 0.15 µm | output NOR |

Per-edge jitter 0.23 / 0.61 / 1.15 ps at 3.3 / 1.0 / 0.4 V pp; system at 3.3 V pp 1.95 / 1.26 ps, L(300 kHz)
−111.0 / −120.7 dBc/Hz (FFNC off / on). Full report: `docs/design/step76_refbuf.md`.

### Interview questions — step 76 (27)

<details><summary><b>Q1.</b> What does the reference buffer do in this PLL?</summary>

It turns whatever the crystal oscillator delivers on the REF_IN pad (a 0.4 V pp sine up to a 0–3.3 V square wave) into a clean 1.8 V CMOS edge, refclk, for the clock generator that fires the sub-sampling phase detector. Its own noise is added to the reference, so it is multiplied by N² = 1024 (+30 dB) inside the loop bandwidth.

</details>

<details><summary><b>Q2.</b> Why is reference-buffer noise so important in a sub-sampling PLL?</summary>

The SSPD removes the divider and makes PD/CP noise small, so inside the loop bandwidth the output phase noise is set by the reference path: XO + buffer, times N². In this design FFNC-on L(300 kHz) is reference-limited: a perfect buffer gives −121.3 dBc/Hz, a −155 dBc/Hz buffer −119.7 (with a −160 dBc/Hz clock-generator placeholder).

</details>

<details><summary><b>Q3.</b> How do you convert voltage noise at a threshold crossing into jitter?</summary>

σ_t = σ_v / slope at the crossing. For a sine of V_pp at f the slope at the zero crossing is π f V_pp: 1 V pp at 50 MHz gives 0.157 V/ns, so 63 µV rms of noise is 0.4 ps rms. A square wave with 0.5 ns edges has a 20× steeper slope, so the same noise gives 20× less jitter.

</details>

<details><summary><b>Q4.</b> How do you convert per-edge white jitter into a phase-noise floor?</summary>

Independent jitter σ on every edge at f_ref is sampled white noise: S_φ (single-sided) = 2(2π f_ref σ)²/f_ref, so L = (2π f_ref σ)²/f_ref. 0.4 ps at 50 MHz is −155 dBc/Hz; 0.61 ps is −151.4 dBc/Hz.

</details>

<details><summary><b>Q5.</b> Why AC coupling at the input?</summary>

The external oscillator may have any DC level (0–1.8 V CMOS, 0–3.3 V CMOS, a clipped sine around ground or an AC-coupled sine). The coupling capacitor removes the DC, and the receiver biases its own input at its switching point, so the crossing is always at the middle of the input waveform and the duty cycle stays near 50 %.

</details>

<details><summary><b>Q6.</b> How is the receiver input biased?</summary>

Through R_B = 5 MΩ from the output of the first stage to its input. At DC this is a short, so the inverter sits at its trip point: v_in = v_s1 ≈ 0.86 V. At 50 MHz R_B is an open circuit, so the full signal reaches the gate.

</details>

<details><summary><b>Q7.</b> What is the pad divider for?</summary>

Two 500 kΩ poly resistors hold the pad at VDDA/2 when the oscillator is AC-coupled externally or not connected, so the pad never floats. They load the oscillator with 250 kΩ, negligible next to C_AC's 0.9 kΩ at 50 MHz, and cost 3.3 µA.

</details>

<details><summary><b>Q8.</b> Why is the first stage a current-limited inverter and not a plain self-biased inverter?</summary>

A plain inverter at VDDA/2 runs both devices in strong inversion with a current that changes 4–6× over VDDA and process. The header current source fixes the current (60 µA) and lets the wide input devices run in moderate inversion, where g_m/I is high (~13 V⁻¹): more g_m — less noise — per µA.

</details>

<details><summary><b>Q9.</b> What is C_H for?</summary>

The header is a current source, so the pMOS input device's source sees a high impedance and is degenerated: it would add almost no g_m. C_H (1.8 pF MIM) ties that node to VDDA at high frequency, so XP1 adds its g_m (current reuse) and the header's own noise current is shunted into the capacitor instead of the output. It lowered the white input noise by about 30 %.

</details>

<details><summary><b>Q10.</b> Why is XP1's body tied to its source?</summary>

Its source sits about 0.9 V below VDDA. With the body at VDDA its threshold rose to 1.24 V (body effect), costing headroom at VDDA = 2.97 V. In its own n-well with body = source, V_th is about 1.0 V.

</details>

<details><summary><b>Q11.</b> Why thick-oxide devices in S1 and S2?</summary>

They connect to VDDA (3.3 V) and S1's gate connects to the pad through C_AC: with a 3.3 V pp input the gate swings from about −0.8 V to 2.5 V. Thin-oxide devices are rated for 1.8 V. The project rule: 3.3 V devices wherever a node touches VDDA or a pad.

</details>

<details><summary><b>Q12.</b> How is the signal brought from 3.3 V down to 1.8 V?</summary>

A thick-oxide inverter powered from VDDL (1.8 V): its gate tolerates the 0–3.3 V swing of S2, its output swings 0–1.8 V. A down-shift needs no latch (unlike the up-shift of step 75) because a 3.3 V input fully turns the 1.8 V inverter on and off.

</details>

<details><summary><b>Q13.</b> Why is the output a NOR gate?</summary>

NOR(s3, not pll_en): when enabled it is an inverter, when disabled it forces refclk low, so the clock generator and SSPD see a defined, quiet state. All controls default to 0.

</details>

<details><summary><b>Q14.</b> What happens when pll_en = 0?</summary>

XPEN pulls the header gate to VDDA (S1 current off), XNS1 pulls s1 low (and through R_B the input), so no device carries static current; the output NOR holds refclk at 0. Only the pad divider (3.3 µA) remains.

</details>

<details><summary><b>Q15.</b> Why is there a start-up time after pll_en?</summary>

When enabled, the input sits at 0 V and must charge to the trip point through R_B into C_AC + C_in before the crossing is centred; simulated, the periods are within 2 % about 4 µs after pll_en and the bias keeps settling for some 30 µs. The CCO filter needs about 0.3 ms, so the FLL (step 81) waits for that anyway.

</details>

<details><summary><b>Q16.</b> How did you choose R_B?</summary>

Its noise current flows into the input node. Inside the bias loop the low-frequency level at the gate is 4kT·R_B/(1 + A)², flat up to the corner (1 + A)/(2π R_B C), which lies above 300 kHz because A ≈ 190. So a smaller R_B gives LESS noise in the 10 kHz–1 MHz band: 20 MΩ gave 3.3 nV/√Hz at the gate, 5 MΩ 1.7 nV/√Hz. 5 MΩ is also four times smaller (10 segments) and starts up faster. The lower limit: the input high-pass corner, about 1.5 MHz, must stay far below 45 MHz.

</details>

<details><summary><b>Q17.</b> How was the noise verified?</summary>

Two methods. M14: .noise at the self-bias point (which is the crossing point of S1) gives the input-referred spectrum, including flicker and R_B; its low-frequency part converts to phase noise through the input slope. M15: a transient with the white part of that spectrum injected at the pad (trnoise) gives the per-edge jitter directly from 300 edge times, including the stage's non-linear switching.

</details>

<details><summary><b>Q18.</b> Why not just divide the integrated .noise output by the output slope?</summary>

S1 is a high-gain comparator: its small-signal low-frequency gain (≈ 190) is much larger than the gain that applies to the fast-moving input during a crossing (≈ 30), so output-referred noise over output slope overestimated the jitter 2×. The transient-noise run measures what really happens; the small-signal analysis is used only for the slow (quasi-static) noise, where it is exact.

</details>

<details><summary><b>Q19.</b> What is the effective noise bandwidth of the receiver and why does it matter?</summary>

σ_v = σ_t × slope; with the white input noise S_w this defines B_eff = σ_v²/S_w ≈ 650 MHz, close to S1's gain-bandwidth. The per-edge jitter is √(S_w·B_eff)/slope: lower noise density (more g_m) or less bandwidth reduce it, at the price of power or delay.

</details>

<details><summary><b>Q20.</b> What does the receiver achieve?</summary>

Per-edge jitter 0.23 ps with a 3.3 V pp input (−159.9 dBc/Hz), 0.61 ps at 1 V pp (−151.4), 1.15 ps at 0.4 V pp (−145.8). With the ring, CCO and loop: 1.95 / 1.26 ps and L(300 kHz) −111.0 / −120.7 dBc/Hz at 3.3 V pp; at 1 V pp 1.97 / 1.33 ps and −110.6 / −118.2.

</details>

<details><summary><b>Q21.</b> Does the design meet the specification?</summary>

All guaranteed (max) limits are met over the whole 0.4–3.3 V pp input range. The typical targets need a full-swing input (≥ about 2 V pp, e.g. a 3.3 V CMOS crystal oscillator): with a 1 V pp sine the FFNC-on L(300 kHz) is 1.8 dB above its typical target. So the typical specification is stated for a full-swing reference.

</details>

<details><summary><b>Q22.</b> Why not simply burn more current to fix the 1 V pp case?</summary>

Doubling S1's current (+0.2 mW) lowers the white jitter only by √2: L(300 kHz) FFNC-on would still be about −119.5 dBc/Hz at 1 V pp, while the worst-case power margin (0.4 mW) would be almost used up. A full-swing XO costs nothing on chip.

</details>

<details><summary><b>Q23.</b> How sensitive is the edge time to supply noise?</summary>

Measured by stepping VDDA and VDDL by 10 mV and comparing the delays (finite difference). Supply ripple at the reference rate moves every edge the same way and appears as a spur; random supply noise inside the loop bandwidth is multiplied by N² like any reference noise.

</details>

<details><summary><b>Q24.</b> What sets the duty cycle and does it matter?</summary>

The input is centred by the self-bias, and later stages switch near mid-supply, so the duty stays 47–53 % at typical supplies; at VDDA = 3.63 V with the smallest (0.4 V pp) input it rises to about 65 % because S2's threshold moves with VDDA while s1's edges are slow. The SSPD only uses the rising edge (the clock generator makes its own pulses from it), but the proposal guarantees ≥ 45 %.

</details>

<details><summary><b>Q25.</b> Which layout precautions does this block need?</summary>

REF_IN routed short and shielded from the pad, the high-impedance vin node (5 MΩ) guarded and away from digital lines, C_AC with its bottom plate on the pad side (bottom-plate parasitic driven by the low-impedance source), XN1 and XP1 with dummies, XP1 in its own n-well, C_H next to the header, R_B as a serpentine over a quiet area.

</details>

<details><summary><b>Q26.</b> Why the bottom plate of C_AC on the pad side?</summary>

The MIM bottom plate has a larger parasitic capacitance to the substrate. On the pad side it is driven by the 50 Ω source and does not matter; on vin it would attenuate the signal and couple substrate noise into a 20 MΩ node.

</details>

<details><summary><b>Q27.</b> What ESD considerations apply?</summary>

The pad cell provides primary ESD protection. Behind it the only connections are poly resistors and the MIM capacitor's bottom plate — no gate is connected directly to the pad. The MIM dielectric must withstand the pad's ESD voltage; this is checked with the pad cell in the top-level ESD review.

</details>

<a id='step-77'></a>
## Step 77 — Clock generator: sampling clocks, hold pulse, Gm pulse

*Roadmap step 77: schematic*

### What happens
Every reference period the sub-sampling phase detector does three things in a fixed order: it **samples** the VCO on
C1 (the sampling switch opens), **converts** the sample into charge with a short Gm current pulse of width $\tau$ (the Gm
reads C1), and then **transfers** the sample to C2 for the NCB (the hold switch closes for $W_H$). *(Order revised in
step 78: reading C2 after the C1–C2 charge sharing would have put a pole z = 0.5 into the loop.)* The clock generator turns the single `refclk` edge into
these three phases — three complementary pairs cks/cksb, ckh/ckhb, ckg/ckgb — and guarantees that they never overlap.

### How it happens
Inverter chains make the small delays (the non-overlap gaps); an **RC pulser** makes each pulse. Its output is the
input AND its own R-C-delayed copy, so a rising edge gives one pulse as long as the RC delay plus the gates:
$$\text{out} = \overline{\overline{\text{in}} + \overline{\text{in}_{RC}}} = \text{in}\cdot\text{in}_{RC},\qquad
\tau \approx \ln 2\,R\,C + t_{gates}$$
The hold pulse is fired by the *end* of the Gm pulse through a small NOR latch (set by ckg, reset while refclk is
low), so the order can never break and the hold pulser's RC is reset for ~10 ns every period. The loop sees the delay from the
sampling instant to the middle of the Gm pulse and the phase-detector gain set by the pulse width:
$$t_p = t_{gap1} + \frac{\tau}{2},\qquad \beta = 2A\,g_m\frac{\tau}{T}$$
Only the sampling edge carries noise into the loop (the later edges act on a sample already taken). Its per-edge
jitter makes a white floor at the reference; in a transient we reproduce each inverter's noise with a trnoise current
of amplitude NA:
$$\mathcal{L}_{floor} = \frac{(2\pi f_{ref}\,\sigma_t)^2}{f_{ref}},\qquad NA = \sqrt{\frac{S_i}{2\,NT}}$$

### Why it happens this way
The sampling path is the shortest (5 stages) with the strongest drivers because it is the only one whose noise reaches
the output. $\tau$ is set mostly by gates, not by the resistor: gate delay grows when transistors are slow — exactly
when the ring VCO's gain $K_{VCO}$ falls — so $\beta K_{VCO}$ is compensated (×0.86–1.21) although $\tau$ alone spreads
106–408 ps. When `pll_en` = 0 refclk is held low: cks = 1 (track), no pulses — the block needs no enable pin.

**Methods used:** M16 (large-signal transient over 21 conditions incl. the combined MOS + R/C corners), M17 (transient
noise with per-node noise currents at the trip level), M9 (system budget), M4/M5 (LVS, drawing checks). `docs/methods.md`.

**Figure 77.1 — the three phases at tt, 27 °C.** Zoom on the first nanoseconds to see the gaps.

In [ ]:
import json
import numpy as np
from IPython.display import Image, display
D = lambda n: json.load(open("../design/clkgen/data/" + n))
sim, wv = D("step77_sim.json"), D("step77_wave.json")

t = np.array(wv["t"])*1e9
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.03)
for i, (k, c) in enumerate((("refclk", "#59636e"), ("cks", "#2a78d6"), ("ckh", "#1baf7a"), ("ckg", "#eb6834"))):
    fig.add_trace(go.Scatter(x=t, y=wv[k], name=k, line=dict(color=c, width=1.6)), row=i + 1, col=1)
    fig.update_yaxes(title_text=k + " (V)", row=i + 1, col=1)
fig.update_xaxes(title_text="time (ns)", row=4, col=1); fig.update_layout(height=560)
show(fig)

**Figure 77.2 — timing over 21 conditions** (corners, temperature, VDDL, f_ref, duty, refclk edge). Dashed lines: requirements. Hover for the condition.

In [ ]:
import json
import numpy as np
from IPython.display import Image, display
D = lambda n: json.load(open("../design/clkgen/data/" + n))
sim, wv = D("step77_sim.json"), D("step77_wave.json")

T = sim["timing"]
lab = [f"{a['corner']} {a['temp']}°C {a['vdd']} V {a['f']/1e6:.0f} MHz duty {a['duty']} tr {a['tr']*1e12:.0f} ps" for a in T]
fig = make_subplots(rows=1, cols=4, subplot_titles=("τ (ps)", "W_H (ns)", "t_p (ns)", "min gap (ps)"))
for i, (f, req) in enumerate(((lambda a: a["tau"]*1e12, None), (lambda a: a["W_H"]*1e9, 1.5),
                              (lambda a: a["t_p"]*1e9, 5), (lambda a: min(a["gap1"], a["gap2"])*1e12, 100))):
    fig.add_trace(go.Scatter(x=list(range(len(T))), y=[f(a) for a in T], mode="markers", text=lab,
                             marker=dict(color="#2a78d6", size=7), showlegend=False), row=1, col=i + 1)
    if req: fig.add_hline(y=req, line_dash="dot", line_color="#eb6834", row=1, col=i + 1)
fig.update_xaxes(title_text="condition #"); fig.update_layout(height=380)
show(fig)

**Figure 77.3 — τ·K_VCO compensation.** τ from this step, K_VCO from step 74 (relative to tt). The loop needs the product, not τ alone. Hover the bars.

In [ ]:
import json
import numpy as np
from IPython.display import Image, display
D = lambda n: json.load(open("../design/clkgen/data/" + n))
sim, wv = D("step77_sim.json"), D("step77_wave.json")

T = {f"{a['corner']} {a['temp']}": a["tau"] for a in sim["timing"] if a["f"] == 5e7 and a["duty"] == 0.5 and a["tr"] == 3e-10}
tn = sim["timing"][0]["tau"]
rows = [("ss 110", 0.684), ("ss_hh 110", 0.684), ("tt 27", 1.0), ("ff -40", 1.485), ("ff_ll -40", 1.485)]
x = [r[0] + " °C" for r in rows]; tr = [T[r[0]]/tn for r in rows]; kr = [r[1] for r in rows]
fig = go.Figure([go.Bar(x=x, y=tr, name="τ / τ_tt", marker_color="#59636e"),
                 go.Bar(x=x, y=kr, name="K_VCO / K_tt", marker_color="#1baf7a"),
                 go.Bar(x=x, y=[a*b for a, b in zip(tr, kr)], name="product", marker_color="#2a78d6")])
fig.add_hrect(y0=0.75, y1=1.25, fillcolor="#2a78d6", opacity=0.07, line_width=0)
fig.update_layout(barmode="group", height=400, yaxis_title="relative to tt", bargap=0.25)
show(fig)

**Figure 77.4 — jitter of the sampling edge (method M17).** Each inverter's trip-point noise current, the
measured σ and the floor it makes at 50 MHz.

In [ ]:
import json
import numpy as np
from IPython.display import Image, display
D = lambda n: json.load(open("../design/clkgen/data/" + n))
sim, wv = D("step77_sim.json"), D("step77_wave.json")

j = sim["jitter"]
print("noise current at the trip point:", {k: f"{np.sqrt(v)*1e12:.2f} pA/rtHz" for k, v in j["S"].items()})
print(f"edges {j['n']}: sigma raw {j['sigma_raw']*1e12:.2f} ps (numerical drift), detrended {j['sigma']*1e12:.3f} ps, "
      f"noiseless baseline {j['baseline']*1e12:.4f} ps")
print(f"floor at 50 MHz: {j['L_floor']:.1f} dBc/Hz (allocation -160)")
s = np.logspace(-14, -11.5, 60); L = 10*np.log10((2*np.pi*5e7*s)**2/5e7)
fig = go.Figure([go.Scatter(x=s*1e12, y=L, name="L = (2π f σ)² / f", line=dict(color="#2a78d6")),
                 go.Scatter(x=[j["sigma"]*1e12], y=[j["L_floor"]], mode="markers", name="clock generator",
                            marker=dict(size=11, color="#eb6834"))])
fig.add_hline(y=-160, line_dash="dot", line_color="#e87ba4")
fig.update_xaxes(type="log", title_text="per-edge jitter σ (ps)"); fig.update_yaxes(title_text="floor (dBc/Hz)")
fig.update_layout(height=380); show(fig)

**The schematics** (top → RC pulser → complementary driver; LVS-clean against the simulated netlist).

In [ ]:
from IPython.display import Image, display
for n in ("clkgen", "ckg_pulse_h", "ckg_hlatch", "ckg_drv"):
    print("sspll_" + n); display(Image(filename=f"../docs/img/step77_sch_{n}.png", width=900))

### Sizing table and results
| Device | Model | W per finger × nf | L | Role |
|---|---|---|---|---|
| INVS (n / p) | nfet_01v8 / pfet_01v8 | 0.42 / 0.84 × 1 | 0.15 µm | gap and pulser inverters |
| INVM (n / p) | nfet_01v8 / pfet_01v8 | 1 / 2 × 1 | 0.15 µm | refclk buffer, driver middle stage |
| INVL (n / p) | nfet_01v8 / pfet_01v8 | 2 / 4 × 2 | 0.15 µm | driver outputs |
| NOR2 (n / p) | nfet_01v8 / pfet_01v8 | 0.84 / 1.68 × 1 | 0.15 µm | pulser gates, 3-NOR hold latch |
| R_H / C_H | res_xhigh_po_0p35 / cap_mim_m3_1 | 0.35 × 6 µm / 6 × 6 µm | — | 34.9 kΩ, 76 fF: hold pulse 2.15 ns |
| R_G / C_G | res_xhigh_po_0p35 / cap_mim_m3_1 | 0.35 × 0.5 µm / 3.5 × 3.5 µm | — | 3.48 kΩ, 27 fF: Gm pulse 202 ps |

tt: $t_s$ 228 ps, gaps 171 / 318 ps, $\tau$ 200 ps, $W_H$ 2.16 ns, $t_p$ 0.28 ns, 82 µW, jitter 0.15 ps (−163.4 dBc/Hz).
Worst: $t_p$ 0.50 ns, $W_H$ 1.51 ns, gap 110 ps. (Revised in step 78; the first version had $t_p$ = 2.63 ns.) System (3.3 V pp): 1.963 / 1.254 ps, L(300 kHz) −110.9 / −121.0 dBc/Hz.
Full report: `docs/design/step77_clkgen.md`; testbenches: `docs/testbench_guide.md`, step 77.

### Interview questions — step 77 (27)

<details><summary><b>Q1.</b> What does the clock generator do in this sub-sampling PLL?</summary>

From one reference edge (refclk) it makes three complementary control pairs, in a fixed order: cks/cksb for the sampling switch (it tracks the VCO and opens at the reference edge), ckg/ckgb for the short current pulse of the pulsed-Gm charge pump, which reads C1 right after the sample, and ckh/ckhb for the hold switch that then shares C1 onto C2 for the NCB. It is the timing skeleton of the SSPD + charge pump (order revised in step 78).

</details>

<details><summary><b>Q2.</b> Why does the sampling happen on the falling edge of cks and not on a rising edge?</summary>

While cks = 1 the switch conducts and the sampling capacitor follows the VCO (track). The sample is the voltage at the moment the switch opens, i.e. when cks falls. The reference's rising edge is the timing reference, so cks must fall in response to refclk rising: the driver gives cks from the inverted path.

</details>

<details><summary><b>Q3.</b> Why are there gaps between the three phases?</summary>

Two switches must never conduct at the same time, and the Gm must read a settled sample. If the Gm current flowed while sw1 still tracked the VCO, it would see the VCO instead of the sample; if sw2 shared C1 with C2 while the Gm current flowed, the current would follow the sharing transient. Gap 1 (cks fall → ckg rise) and gap 2 (ckg fall → ckh rise) are the non-overlap times: 171 / 318 ps typical, at least 110 ps over all corners (requirement ≥ 100 ps).

</details>

<details><summary><b>Q4.</b> How does an RC pulser make a pulse from an edge?</summary>

The input and a delayed, inverted copy of it go into a gate (here a NOR of the inverted input and the inverted delayed input, which is input AND NOT delayed). After a rising edge both conditions are true until the delayed copy has also switched, so the output is high for one delay: the pulse width is the delay of the R-C stage plus the gates. A falling edge produces no pulse.

</details>

<details><summary><b>Q5.</b> What sets the width of the RC pulse?</summary>

The RC node charges exponentially and the next inverter switches when it crosses its trip point (about half the supply), so the RC part is ≈ ln2 · R·C. To it add the delays of the inverters and the NOR. For the hold pulse RC dominates (R_H ≈ 34 kΩ, C_H ≈ 75 fF → ≈ 1.8 ns of the 2.15 ns); for the 200 ps Gm pulse the gates dominate (minimum-length resistor).

</details>

<details><summary><b>Q6.</b> Why does the Gm pulse width τ matter so much?</summary>

The charge pump delivers a charge g_m · v_sample · τ per reference period, so the phase-detector gain is β = 2A·g_m·τ/T. The loop bandwidth and phase margin depend on β·K_VCO. A τ that varies ±50 % over PVT moves the bandwidth by ±50 % unless something else compensates.

</details>

<details><summary><b>Q7.</b> τ varies from 106 to 408 ps over corners. Why is that acceptable?</summary>

Because the loop needs β·K_VCO, not τ alone. τ is long when transistors are slow (ss, cold, high R) and the VCO gain is then small; τ is short in ff where K_VCO is large. The product τ·K_VCO stays within ×0.86–1.21 of nominal in the worst conditions found. The final check of β·K_VCO with g_m and A is done in step 79.

</details>

<details><summary><b>Q8.</b> Why not use a high-precision resistor for τ?</summary>

A poly resistor's temperature coefficient (xhigh_po −19 % over −40…110 °C) and its process corner (±20 % or more) add to the gate-delay variation. Making R small and letting the gates set most of τ lets the τ variation track the ring VCO's transistors, which is the compensation we want. Absolute accuracy would need a trimmed or locked delay (a DLL), which costs power and area for no system benefit here.

</details>

<details><summary><b>Q9.</b> What are the SKY130 'combined' corners and why did you use them here?</summary>

The PDK has MOS corners (tt, ss, ff, sf, fs) and resistor/capacitor corners. The combined library sections such as ss_hh (slow MOS, high R, high C) and ff_ll move both together, and hl / lh move R and C the opposite way. A circuit whose timing is R·C plus gate delay has its extremes there, not at plain ss/ff: τ's worst cases came from ss_hh (408 ps) and ff_ll (106 ps).

</details>

<details><summary><b>Q10.</b> What is t_p and why must it be ≤ 5 ns?</summary>

t_p is the delay from the sampling instant to the centre of the Gm current pulse: the loop's transport delay. In the loop gain it appears as e^{−s·t_p}, i.e. a phase lag ω_c·t_p at the crossover. Step 73 allowed up to 5 ns for ≤ 3° of phase-margin loss. With the Gm right after the sample t_p = 0.28 ns typical and 0.50 ns worst (it was 2.63 ns when the Gm came after the hold pulse).

</details>

<details><summary><b>Q11.</b> Why must W_H be at least 1.5 ns?</summary>

During the hold pulse the charge on C1 is shared with C2 through sw2. The transfer must settle (several R_on·C time constants) so that C2, which the NCB reads for the whole next period, holds the final shared value, and some time is reserved for the switch's own charge injection to die away. 1.5 ns is the requirement; the minimum over corners is 1.51 ns (ff_ll, 110 °C).

</details>

<details><summary><b>Q12.</b> Why does the hold pulse width depend on the reference duty cycle?</summary>

The RC node of the hold pulser is reset while its trigger is low, i.e. while refclk is low. With a short low time (duty 65 %) the node is not fully discharged at the next edge, so it reaches the trip point earlier and the pulse shrinks (1.98 ns at 65 % vs 2.22 ns at 35 %). It stays above 1.5 ns; a reset nMOS on the RC node would remove the dependence if needed.

</details>

<details><summary><b>Q13.</b> How is the clock generator disabled when the PLL is off?</summary>

It has no enable pin. When pll_en = 0 the reference buffer holds refclk at 0: cks stays 1 (the sampling switch tracks), and no hold or Gm pulses are produced, so the charge pump injects no charge and the circuit draws only leakage.

</details>

<details><summary><b>Q14.</b> How much power does it use and why so little?</summary>

78 µW typical (69–108 µW over corners). It is CMOS logic switching once per reference period: P ≈ C_total·V²·f with tens of fF per node at 50 MHz. It fits inside the 0.12 mA allocation for the SSPD and clock generator.

</details>

<details><summary><b>Q15.</b> Why does the jitter of the clock generator matter?</summary>

The sampling edge is the PLL's time reference. Any random delay added between refclk and the moment the switch opens is reference jitter and is multiplied by N = 32 (+30 dB) inside the loop bandwidth, exactly like the crystal's own noise. The Gm and hold edges do not matter for noise: they act after the sample has been taken.

</details>

<details><summary><b>Q16.</b> How did you estimate the jitter of an inverter chain in ngspice?</summary>

Method M17: .noise on each inverter size biased at its trip point gives the noise current it makes while switching (2.4–7.2 pA/√Hz). In a transient, every node of the sampling path gets a white trnoise current of that size, with NA = √(S/(2NT)). 123 edges of a noiseless 500 MHz input give the spread of the output edge times: 0.15 ps rms.

</details>

<details><summary><b>Q17.</b> Why inject at the trip-point noise level?</summary>

A CMOS inverter is quiet when it rests at a rail (one device off, the other in deep triode with no gain). It contributes to timing only while it passes through the threshold, where both devices conduct with maximum g_m and the node is high-impedance. Using the trip-point density during the whole run is pessimistic, which is fine for a budget check.

</details>

<details><summary><b>Q18.</b> Your first jitter result was 1.17 ps. What was wrong?</summary>

Two things. Some output edges were paired with the wrong input edge, and the simulator's time grid made the edge times drift slowly by about 1.2 ps over the run. A run with all noise scaled by 10⁻⁶ showed the same 1.2 ps, so it was numerical. Pairing each output edge within 0.6 ns of its input edge and removing a 4th-order trend gave 0.15 ps, with a noiseless baseline of 0.0002 ps.

</details>

<details><summary><b>Q19.</b> How do you turn 0.15 ps of per-edge jitter into phase noise?</summary>

Independent jitter on each 50 MHz edge is a white phase-noise floor L = (2π·f_ref·σ)²/f_ref = (2π·5×10⁷·1.5×10⁻¹³)² / 5×10⁷ = 4.4×10⁻¹⁷ → −163.4 dBc/Hz. The step-72 allocation was −160 dBc/Hz, so there is 3.4 dB of margin.

</details>

<details><summary><b>Q20.</b> Why are the output drivers bigger than the rest of the logic?</summary>

They drive the switch gates (30 / 20 fF) with sharp edges. A slow edge on the sampling switch makes the sampling instant depend on the signal level (the switch turns off when V_GS crosses V_T, which depends on the sampled voltage) and makes the edge more sensitive to noise. 4 / 8 µm output stages give 76 ps rise times.

</details>

<details><summary><b>Q21.</b> Why are cks and cksb generated by paths of 2 and 3 inverters, and is the skew a problem?</summary>

One output needs the inverted signal, so its path has one more stage. The extra stage gives about 12 ps skew. For a complementary transmission gate, a small skew only changes which device turns off last; the sampling instant is set by the last one to turn off, and it is the same every period, so it adds no jitter and only a fixed delay.

</details>

<details><summary><b>Q22.</b> Which requirements does step 77 hand on to later steps?</summary>

Loads ≤ 30 fF on cks/cksb and ≤ 20 fF on the others (switch sizes in steps 78/79); the Gm switch must handle pulses as short as 106 ps; β·K_VCO must be checked over corners with the real g_m and A (step 79); the layout must keep the sampling path short and symmetric (parasitic delay adds to t_p and to jitter).

</details>

<details><summary><b>Q23.</b> Why was the order of the Gm and hold pulses changed in step 78?</summary>

Charge sharing makes C2 a memory: v2[k] = (C1 v1[k] + C2 v2[k−1])/(C1 + C2), a first-order low-pass with a pole at z = C2/(C1 + C2) = 0.5. If the Gm read C2 (or C1 after the sharing), that pole would sit inside the PLL loop and cost about 16° of phase margin at f_c = f_ref/20; step 73 did not include it. Reading C1 before the sharing gives the Gm the fresh sample; the NCB, which reads C2, keeps the pole that the FFNC model already contains.

</details>

<details><summary><b>Q24.</b> Why does the hold pulser need a latch as its trigger?</summary>

The hold pulser's RC node (R_H·C_H ≈ 2.6 ns) is reset only while its input is low. Triggered directly by the end of the 200 ps Gm pulse, its input would be low for only 200 ps and the pulser would never fire. The latch is set by ckg and reset while refclk is low (cks = 1); its output q·NOT(ckg) rises when the Gm pulse ends and stays high until refclk falls, so the RC node gets ~10 ns to reset every period.

</details>

<details><summary><b>Q25.</b> How would you check this block in Cadence?</summary>

Same testbench: an ideal 50 MHz pulse on refclk, capacitive loads, tran with the Calculator's cross() and delay() for the edges, Corners setup with the combined MOS+R/C sections. Jitter by Spectre transient noise (noisefmax ≈ 500 GHz) or, better, pss + pnoise with sampled (jitter) output at the cks crossing.

</details>

<details><summary><b>Q26.</b> What would you change if τ had to be accurate to ±10 %?</summary>

Replace the RC pulser by a delay that is locked to the reference: e.g. a current-starved delay biased from a replica delay line in a DLL, or trim the R with a few switched segments after measurement. Both cost area and power; with this loop the natural τ·K_VCO compensation makes them unnecessary.

</details>

<details><summary><b>Q27.</b> Why are all the devices thin-oxide 1.8 V transistors at minimum length?</summary>

The block runs from VDDL = 1.8 V and has to be fast: minimum L gives the smallest gate capacitance and delay, and the power is dominated by switching, not leakage. Matching is not important here because the absolute delays are re-referenced every period and only their noise matters.

</details>

<a id='step-78'></a>
## Step 78 — SSPD with hold: sampler, dummy sampler, isolation buffers

*Roadmap step 78: schematic*

### What happens
At every reference edge the sub-sampling phase detector freezes the VCO waveform on a capacitor C1. Near lock the
frozen voltage is proportional to the time between the VCO zero crossing and the reference edge — that is the phase
error, measured with no divider. Right after the sample the pulsed Gm (step 79) reads v1p − v1m; then the sample is
shared onto C2, which holds it for a whole period for the NCB.

### How it happens
Per side: the VCO output (clkp) → two **isolation inverters** → a series resistor $R_s$ → node f → the **sampling
switch** onto C1 (35 fF) and, on the opposite phase, a **dummy sampler** onto CD. $R_s$ and C1 round the square into
a near-sine whose slope at the crossing is exactly $(V_{DD}/2)/\tau_s$, so the gain per side in volts per radian is
$$A = \frac{1}{\omega_0}\left.\frac{dv}{dt}\right|_{cross} = \frac{V_{DD}}{2\,\omega_0\,\tau_s},\qquad
\tau_s = (R_s + R_{buf} + R_{sw})\,C_{1,tot}$$
After the Gm pulse, the hold switch shares C1 with C2, which keeps a memory of the previous value:
$$v_2[k] = (1-\alpha)\,v_1[k] + \alpha\,v_2[k-1],\qquad \alpha = \frac{C_{2,tot}}{C_{1,tot} + C_{2,tot}}$$
The sampled kT/C noise, divided by the gain, is a white phase floor that is **not** multiplied by N²:
$$\mathcal{L}_{kT/C} = \frac{kT}{2\,C\,A^2 f_{ref}}$$

### Why it happens this way
- The RC near-sine gives a gain set by R·C with a linear range of ±0.7 rad; a raw square would give an
  inverter-dependent slope linear over picoseconds. kT/C does not depend on R.
- **The Gm reads C1, not C2.** Reading after the sharing would put the pole z = α ≈ 0.5 inside the PLL loop
  (16° of phase lag at f_c = f_ref/20). Step 77's clock order was changed for this.
- The dummy keeps the buffer's load — and its supply current — the same in every phase: no ripple at f_ref on the LDO.
- 35 fF instead of 100 fF saves a third of the power; kT/C stays 15 dB under the reference at the output.

**Methods used:** M18 (beat-frequency characteristic), M19 (kick-back by difference of two transients), M17 (isolation
buffer jitter), M9 (system budget), M4/M5 (LVS, drawing checks). `docs/methods.md`.

**Figure 78.1 — the sampled characteristic** (beat run, tt). Slider: how far from the lock point the line fit reaches; the slope of the fit is 2A.

In [ ]:
import json
import numpy as np
from IPython.display import Image, display
D = lambda n: json.load(open("../design/sspd/data/" + n))
ch, kk, jj = D("step78_char.json"), D("step78_kick.json"), D("step78_jitter.json")

tt = ch[0]; r = np.array(tt["rows"]); d = (r[:, 1] - tt["phi0"] + np.pi) % (2*np.pi) - np.pi; o = np.argsort(d)
def frame(w):
    m = np.abs(d) < w; s = np.polyfit(d[m], r[m, 2], 1)[0]
    x = np.linspace(-w, w, 2)
    return ([dict(x=d[o], y=r[o, 2], name="v1p − v1m (Gm reads this)", mode="lines+markers", line=dict(color="#2a78d6")),
             dict(x=d[o], y=r[o, 3], name="v2p − v2m (held for the NCB)", mode="markers", marker=dict(color="#eb6834", size=6)),
             dict(x=x, y=s*x, name="line fit", line=dict(color="#1f2328", dash="dash"))],
            f"fit over ±{w} rad: 2A = {s:.3f} V/rad → A = {s/2:.3f} V (the gain falls as the fit leaves the linear part)")
def lay(fig):
    fig.update_xaxes(title_text="VCO phase at the sample, relative to lock (rad)"); fig.update_yaxes(title_text="V")
show(sweep([0.2, 0.35, 0.7, 1.0, 1.5], frame, "fit half-width (rad)", fmt="{}", active=1, layout=lay))

**Figure 78.2 — A, common mode and power over 17 conditions** (combined MOS + R/C corners). Hover for the condition.

In [ ]:
import json
import numpy as np
from IPython.display import Image, display
D = lambda n: json.load(open("../design/sspd/data/" + n))
ch, kk, jj = D("step78_char.json"), D("step78_kick.json"), D("step78_jitter.json")

lab = [f"{c['corner']} {c['temp']}°C {c['vdd']} V {c['fref']/1e6:.0f} MHz" for c in ch]
fig = make_subplots(rows=1, cols=3, subplot_titles=("A (V/rad); floor 0.35", "common mode of v1 (V)", "power (mW)"))
for i, (k, sc) in enumerate((("A", 1), ("cm", 1), ("power", 1e3))):
    fig.add_trace(go.Scatter(x=list(range(len(ch))), y=[c[k]*sc for c in ch], mode="markers", text=lab,
                             marker=dict(color="#2a78d6", size=8), showlegend=False), row=1, col=i + 1)
fig.add_hline(y=0.35, line_dash="dot", line_color="#eb6834", row=1, col=1)
fig.add_hline(y=0.5, line_dash="dot", line_color="#59636e", row=1, col=1)
fig.update_xaxes(title_text="condition #"); fig.update_layout(height=380)
show(fig)

**Figure 78.3 — kick-back and the dummy sampler (method M19).** Left: how much the sampler moves each clkp edge. Right: SSPD supply current at f_ref.

In [ ]:
import json
import numpy as np
from IPython.display import Image, display
D = lambda n: json.load(open("../design/sspd/data/" + n))
ch, kk, jj = D("step78_char.json"), D("step78_kick.json"), D("step78_jitter.json")

fig = make_subplots(rows=1, cols=2, column_widths=[0.65, 0.35],
                    subplot_titles=(f"clkp edge shift: max {kk['dt_max']*1e15:.1f} fs, spur ≈ {kk['spur_dbc']:.0f} dBc", "f_ref current (µA)"))
fig.add_trace(go.Scatter(x=kk["wave"]["dt_edges"], y=kk["wave"]["dt"], mode="markers", marker=dict(color="#2a78d6", size=6),
                         showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=["with dummy", "without dummy"], y=[kk["I_fref"]*1e6, kk["I_fref_nodummy"]*1e6],
                     marker_color=["#1baf7a", "#eb6834"], showlegend=False), row=1, col=2)
fig.update_yaxes(type="log", row=1, col=2); fig.update_xaxes(title_text="time (ns)", row=1, col=1)
fig.update_yaxes(title_text="fs", row=1, col=1); fig.update_layout(height=360)
show(fig)
print(f"isolation-buffer jitter (M17): {jj['sigma']*1e12:.3f} ps per side, {jj['sigma']/np.sqrt(2)*1e12:.3f} ps differential, "
      f"baseline {jj['baseline']*1e12:.4f} ps")

**Figure 78.4 — what kT/C costs.** Slider: the sampling capacitance. The floor is far below the reference contribution (×N²) at the output.

In [ ]:
import json
import numpy as np
from IPython.display import Image, display
D = lambda n: json.load(open("../design/sspd/data/" + n))
ch, kk, jj = D("step78_char.json"), D("step78_kick.json"), D("step78_jitter.json")

kT = 1.380649e-23*300; A = ch[0]["A"]
def frame(C):
    C = C*1e-15; L = 10*np.log10(kT/(2*C*A**2*50e6)); f = np.logspace(4, 7.4, 50)
    return ([dict(x=f, y=[L]*len(f), name="SSPD kT/C (not ×N²)", line=dict(color="#2a78d6")),
             dict(x=f, y=[-163.4 + 30.1]*len(f), name="clock-generator edge ×N²", line=dict(color="#eb6834", dash="dot"))],
            f"C = {C*1e15:.0f} fF: kT/C floor {L:.1f} dBc/Hz, {(-133.3) - L:.1f} dB under the clock generator alone")
def lay(fig):
    fig.update_xaxes(type="log", title_text="offset (Hz)"); fig.update_yaxes(title_text="dBc/Hz at the output", range=[-155, -125])
show(sweep([20, 35, 46, 70, 100], frame, "C (fF)", fmt="{}", active=2, layout=lay))

**The schematics** (top → one side → transmission gate; LVS-clean against the simulated netlist).

In [ ]:
from IPython.display import Image, display
for n in ("sspd", "sspd_side", "sspd_tg"):
    print("sspll_" + n); display(Image(filename=f"../docs/img/step78_sch_{n}.png", width=900))

### Sizing table and results
| Device | Model | W per finger × nf | L | Role |
|---|---|---|---|---|
| XB1 (n / p) | nfet_01v8 / pfet_01v8 | 0.42 / 0.84 × 1 | 0.15 µm | isolation buffer input (~2 fF on clkp) |
| XB2 (n / p) | nfet_01v8 / pfet_01v8 | 2 / 4 × 1 | 0.15 µm | drives $R_s$ + C |
| XRS | res_xhigh_po_0p69 | 0.69 | 0.6 µm | $R_s$ = 2.16 kΩ |
| sampling / dummy TG | nfet_01v8_lvt / pfet_01v8 | 3 / 1.5 × 1 | 0.15 µm | lvt nMOS: the signal sits mid-rail |
| hold TG | nfet_01v8_lvt / pfet_01v8 | 1 / 1 × 1 | 0.15 µm | C1 → C2 |
| XC1, XC2 / XCD | cap_mim_m3_1 | 4 × 4 / 4.35 × 4.35 µm | — | 34.7 fF / 40.7 fF (= C1 + Gm input) |

A = 0.514 V/rad (0.35–0.77 over corners, ±1.5 % over temperature), α = 0.473, kick-back ≤ 2.3 fs (spur ≤ −134 dBc),
supply current at f_ref 0.76 µA (42 µA without the dummy), buffer jitter 0.080 ps (−169 dBc/Hz), 0.47 mW. System:
1.964 / 1.256 ps, L(300 kHz) −110.9 / −120.9 dBc/Hz. Power is above the step-72 allocation (power review after step 82).
Full report: `docs/design/step78_sspd.md`; testbenches: `docs/testbench_guide.md`, step 78.

### Interview questions — step 78 (27)

<details><summary><b>Q1.</b> What does the SSPD do, in one sentence?</summary>

At every reference edge it takes a sample of the VCO waveform; near lock the sampled voltage is proportional to the time (phase) error between the VCO zero crossing and the reference edge, with gain dv/dt of the waveform, and no divider is needed because sampling a 1.6 GHz wave with 50 MHz folds it down by itself.

</details>

<details><summary><b>Q2.</b> Why is the SSPD gain not divided by N like a PFD's?</summary>

A PFD compares the reference with the VCO divided by N, so a VCO phase error φ appears as φ/N at its input. The SSPD looks at the VCO itself: an error of φ radians of VCO phase moves the sampled voltage by A·φ. The gain is set by the VCO waveform slope, not by N, so the PD and charge-pump noise are not multiplied by N² at the output.

</details>

<details><summary><b>Q3.</b> What is A and how did you get it?</summary>

A is the slope of the sampled waveform at the lock point per side, in volts per radian of VCO phase: dv/dφ = (dv/dt)/ω0. The differential sample v1p − v1m changes by 2A per radian. It is measured with a beat run: the VCO runs at (32 + 1/32)·f_ref, so each reference period samples 1/32 of a VCO cycle further along; a line fit within ±0.35 rad of the zero crossing gives 2A. Result 0.514 V at tt (target 0.5 V).

</details>

<details><summary><b>Q4.</b> Why put a series resistor in front of the sampling capacitor?</summary>

The isolation buffer makes a sharp square wave, whose slope at the crossing would be very large, badly defined (it depends on the inverter edge) and linear over only a few picoseconds. R_s and C1 round it into a near-sine. For an RC-filtered square of amplitude VDD/2 the slope at the crossing is exactly (VDD/2)/τ_s, so A = VDD/(2ω0 R C): set by a resistor and a capacitor, with a linear range of about ±1 rad. kT/C noise does not change: it depends only on C.

</details>

<details><summary><b>Q5.</b> Derive the slope of an RC-filtered square wave at its zero crossing.</summary>

After an edge, v(t) = a(1 − 2e^{−t/τ}/(1 + e^{−T/2τ})) for a square of ±a. It crosses zero where e^{−t_c/τ} = (1 + e^{−T/2τ})/2. The slope is dv/dt = (2a/τ)e^{−t/τ}/(1 + e^{−T/2τ}), which at t_c is exactly a/τ, independent of the period. With a = 0.9 V and A = 0.5 V at 1.6 GHz: τ = a/(Aω0) = 179 ps.

</details>

<details><summary><b>Q6.</b> Why a transmission gate with a low-Vt nMOS as the sampling switch?</summary>

The sample is taken near mid-rail (the common mode of the filtered wave, ≈ 0.72 V). A standard nfet_01v8 with its source at 0.9 V has a body-affected threshold near 1 V and gives 5.4 kΩ at W = 2 µm; the pfet_01v8 is worse (35 kΩ). nfet_01v8_lvt gives 1.8 kΩ at the same size. The pMOS in parallel helps when the waveform is high and cancels part of the nMOS channel charge at turn-off.

</details>

<details><summary><b>Q7.</b> What is the dummy sampler and why is it there?</summary>

A second switch and capacitor (CD = C1 + the Gm input) on the same node f, driven by the opposite clock. When the sampler opens, the dummy closes, so the isolation buffer always drives one capacitor. Its delay and its supply current then do not change at f_ref. Without the dummy the SSPD current has a 42 µA component at f_ref; with it, 0.76 µA (−35 dB). That current would ripple the LDO rail that also feeds the VCO buffers and the NCB: a reference spur.

</details>

<details><summary><b>Q8.</b> What are the isolation buffers for?</summary>

Two inverters between the VCO output (clkp/clkm, shared with the NCB and the divider) and the sampler. The first is small (0.42/0.84 µm), so clkp sees ~2 fF; the second drives R_s and C1. The sampler's switching charge then reaches clkp only through the gate-drain capacitance of the small inverter: the clkp edges move by at most 1.2 fs, a spur estimate of −147 dBc.

</details>

<details><summary><b>Q9.</b> Why does the Gm read C1 and not C2?</summary>

Sharing C1 with C2 makes C2 a memory: v2[k] = (1 − α)v1[k] + αv2[k−1], α = C2/(C1 + C2) ≈ 0.47. A pole at z = α inside the PLL loop would cost about 16° of phase margin at f_c = f_ref/20. So the Gm reads C1 during the 200 ps pulse right after the sample, and only then is C1 shared onto C2. Step 77's clock order was changed for this (t_p also fell from 2.63 to 0.28 ns).

</details>

<details><summary><b>Q10.</b> What is C2 for, then?</summary>

The NCB needs the measured phase error for a whole reference period, while C1 is busy tracking the VCO again. C2 holds it. The charge-sharing pole on that path is the one the FFNC model already contains (−60° at f_ref/7.8 together with the hold and the half-period delay).

</details>

<details><summary><b>Q11.</b> How did you measure α?</summary>

From the same beat run: every period gives a pair (v1[k], v2[k]). A least-squares fit of v2[k] = a·v1[k] + b·v2[k−1] gives b = α = 0.473 and a + b = 1.000 (the hold has unity DC gain). The capacitor ratio predicts (34.7 + 4)/(34.7 + 6 + 3 + 34.7 + 4) ≈ 0.47.

</details>

<details><summary><b>Q12.</b> Why C1 = C2 = 35 fF and not the 100 fF of the roadmap?</summary>

Power. The buffer must swing C at 1.6 GHz: about C·ΔV·f per side. 100 fF cost 0.70 mW, 35 fF 0.47 mW. The kT/C noise goes up by 4.6 dB but it is far below the reference: the system model gives L(300 kHz) −121.00 vs −121.01 dBc/Hz and the same jitter. The step-72 budget already allowed ≥ 50 fF; going lower was checked with the full model.

</details>

<details><summary><b>Q13.</b> How large is the kT/C noise of the sampler in phase terms?</summary>

Each side gets kT/C; the differential sample has 2kT/C. Divided by (2A)² it is a phase variance kT/(2CA²) per sample, a white floor L = kT/(2 C A² f_ref) = −144.7 dBc/Hz with C ≈ 46 fF and A = 0.514 V. It is not multiplied by N² — about 15 dB under the reference contribution at the output.

</details>

<details><summary><b>Q14.</b> Which noise of the SSPD does count like reference noise?</summary>

Timing noise of the isolation buffers. The loop sees the VCO through them, so a random delay there is indistinguishable from reference jitter: it is multiplied by N² inside the bandwidth. The clkp node itself is common with the output path, so its noise is VCO-like. Method M17 on XB1/XB2 gives the jitter of the filtered crossing.

</details>

<details><summary><b>Q15.</b> How does A vary over PVT and why is that acceptable?</summary>

0.35 V (ss_hh) to 0.77 V (ff_ll); over temperature only ±3 % at a fixed corner, over VDDL ±10 % per ±5 %. A follows 1/(R_s C1), so the R/C process corner moves it most. The loop needs β·K_VCO; A·τ·K_VCO at the extreme corners is ×0.69–1.41, inside the icp_trim range (process ×0.55–2.0). What the trim cannot remove is drift, and the drift is small.

</details>

<details><summary><b>Q16.</b> Why does A depend on VDDL?</summary>

The filtered square has amplitude VDD/2 and the slope at the crossing is (VDD/2)/τ_s, so A ∝ VDDL. The LDO keeps VDDL fixed; its tolerance is part of the residual spread handed to step 79.

</details>

<details><summary><b>Q17.</b> What is the common mode of the sample and why is it not 0.9 V?</summary>

0.72 V at tt (0.60–0.82 V over corners). The TG conducts better at low voltage (lvt nMOS) than at high voltage (a standard pMOS), so the falling half of the wave is driven harder: the average and the crossing level sink below VDD/2. Only the differential sample matters for the loop; the Gm input stage (step 79) must accept 0.6–0.85 V.

</details>

<details><summary><b>Q18.</b> What sets the linear range of the SSPD?</summary>

The shape of the sampled waveform. The near-sine has a linear region of about ±0.7 rad around the crossing and peaks at ±1.1 V differential near ±π/2. Outside, the gain falls and reverses: the SSPD has no frequency information, so the FLL must bring the VCO within the capture range first.

</details>

<details><summary><b>Q19.</b> Is the SSPD characteristic symmetric?</summary>

Per side, no: the TG makes rise and fall slightly different. Differentially, yes: the m side is the p side shifted by π, so v1p − v1m is odd-symmetric and both zero crossings have the same slope (0.514 V/rad at tt, checked). The loop polarity (step 79) picks the stable one.

</details>

<details><summary><b>Q20.</b> What is the kick-back problem and how big is it here?</summary>

When the sampling switch opens or closes, its channel charge and the capacitor's sudden connection disturb the node that drives it, and through it the VCO: a periodic disturbance at f_ref, i.e. a reference spur. Here the disturbance reaches clkp through two isolation stages: at most 0.28 mV and 1.2 fs of edge shift; the ring itself is one more buffer away.

</details>

<details><summary><b>Q21.</b> How did you measure the kick-back?</summary>

Two otherwise identical runs at f = 32·f_ref: one with the clock generator running, one with refclk held low (the sampler tracks, nothing switches). The difference of the clkp edge times is the kick-back; its Fourier component at f_ref, as a phase deviation 2π f0 Δt, gives the spur estimate 20 log(a1/2).

</details>

<details><summary><b>Q22.</b> Why must the hold pulse be long (≥ 1.5 ns)?</summary>

sw2 shares C1 and C2 through its on-resistance (~3 kΩ with 1 µm devices): τ ≈ 3 kΩ × 20 fF (C1 and C2 in series) = 60 ps. 2.16 ns is many time constants; the length mainly keeps the end of the transfer far from the next tracking phase and lets the switch's own injection settle.

</details>

<details><summary><b>Q23.</b> What does the Gm input capacitance do to the SSPD?</summary>

It is part of the C1 node: it slows τ_s (lower A) and is copied in the dummy capacitor (CD = C1 + 6 fF) so that the load stays constant. Step 79 must keep its input capacitance near 6 fF per side, or the dummy is resized.

</details>

<details><summary><b>Q24.</b> Why does the hold switch use smaller devices than the sampling switch?</summary>

Its speed is not critical (2 ns to settle 60 ps), and a smaller switch injects less charge onto C2 and loads ckh less.

</details>

<details><summary><b>Q25.</b> How would you verify the SSPD in Cadence?</summary>

Same beat testbench with analogLib vpulse sources and the clock generator. For the characteristic use the Calculator to sample v1p − v1m at the Gm pulse (value at a time list) and plot against phase; or run pss with the reference as the beat frequency and the VCO at 32 + 1/32 times it. Kick-back: two transients and the difference of the cross() times.

</details>

<details><summary><b>Q26.</b> What would you do if a later step needs a larger A?</summary>

Shorten R_s (A ∝ 1/R): R_s = 0.69 × 0.5 µm gives A ≈ 0.55 V. Below that the transistors dominate τ_s and A follows the MOS corners more. Alternatively a smaller C1, at the cost of the Gm input mattering more.

</details>

<details><summary><b>Q27.</b> What layout precautions does the SSPD need?</summary>

Mirror symmetry between the p and m sides; C1, CD and C2 as matched MIM tiles with the bottom plate on VSS; R_s next to the switch; the clock lines cks/cksb routed with equal length to both sides and away from v1; v1 and v2 shielded (they are high-impedance during hold); clkp/clkm entering symmetrically from the VCO buffers.

</details>